-------------------------------------------------------------------------------------------------------
# EUSS Post-Retrofit Measure Packages: MP8, MP9, MP10
-------------------------------------------------------------------------------------------------------
- MP8: Whole Home Electrification (High Efficiency)
- MP9: Whole-Home Electrification + Basic Enclosure Upgrade
- MP10: Whole-Home Electrification + Enhanced Enclosure Upgrade

-------------------------------------------------------------------------------------------------------
# TARE MODEL SCENARIOS
-------------------------------------------------------------------------------------------------------
- Pre-IRA Scenario:
    - NREL End-Use Savings Shapes Database: Measure Package 8/9/10
    - AEO2023 No Inflation Reduction Act
    - Cambium 2021 MidCase
      
- IRA-Reference Scenario:
    - NREL End-Use Savings Shapes Database: Measure Package 8/9/10
    - AEO2023 REFERENCE CASE - HDD and Fuel Price Projections
    - Cambium 2022 and 2023 MidCase

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
from IPython import get_ipython
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Project configuration
from config import PROJECT_ROOT

# Model constants - explicit imports for clarity
from cmu_tare_model.constants import (
    VERBOSE, 
    RCM_MODELS, 
    CR_FUNCTIONS,
    PRINT_DEBUG,
    PRINT_VERBOSE_DATAFRAMES
)
from cmu_tare_model.utils.discounting import (
    PRIVATE_DISCOUNT_RATE_COLS,
    PRIVATE_DISCOUNTING_METHOD_SUFFIXES,
    PUBLIC_DISCOUNTING_METHOD_SUFFIXES
)

# Data loading utility
from cmu_tare_model.utils.load_exported_results_to_df import load_model_run_output

# =============================================================================
# MATPLOTLIB/SEABORN CONFIGURATION
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.close('all')
%matplotlib inline

sns.set_theme(font='sans-serif', style='darkgrid')


# =============================================================================
# PROJECT ROOT AND TIMESTAMP SETUP
# =============================================================================
# Get the current datetime
start_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Format the name of the exported results file using the location ID
result_export_time = datetime.now()
model_run_date_time = result_export_time.strftime("%Y-%m-%d_%H-%M")

print(f"""
PROJECT_ROOT: {PROJECT_ROOT}

Start Time: {start_time}
Model Run Timestamp: {model_run_date_time}

""")

In [ ]:
# Select whether to begin new run or visualize existing model outputs
while True:
    try:
        start_new_model_run = str(input("""
Would you like to begin a new simulation or visualize output results from a previous model run? Please enter one of the following:
Y. I'd like to start a new model run.
N. I'd like to visualize output results from a previous model run.""")).upper()

        print(f"Enter the following input: {start_new_model_run}")

        if start_new_model_run == 'Y':
            print(f"Formatted date for use in file name: {model_run_date_time}")

            print(f"Project root directory: {PROJECT_ROOT}")

            # Relative path to the file from the project root
            relative_path = os.path.join("cmu_tare_model", "model_scenarios", "tare_run_simulation_v2_2.ipynb")

            # Construct the absolute path to the file
            file_path = os.path.join(PROJECT_ROOT, relative_path)
            print(f"File path: {file_path}")

            # Storing Result Outputs in output_results folder
            output_folder_path = os.path.join(PROJECT_ROOT, "cmu_tare_model", "output_results")
            print(f"Result outputs will be exported here: {output_folder_path}")

            # On Windows, to avoid any path-escape quirks, convert backslashes to forward slashes
            file_path = file_path.replace("\\", "/")

            print(f"Running file: {file_path}")

            # iPthon magic command to run a .py file and import variables into the current IPython session
            if os.path.exists(file_path):
                get_ipython().run_line_magic('run', f'-i {file_path}')  # If your path has NO spaces, no quotes needed.
            else:
                print(f"File not found: {file_path}")

            break  # Exit the loop if input is 'Y'
            
        elif start_new_model_run == 'N':
            # Enter the date time of the model run in the following format: YYYY-MM-DD_HH-MM
            model_run_date_time = str(input("Enter the date time of the model run in the following format YYYY-MM-DD_HH-MM: "))
            print(f"Project root directory: {PROJECT_ROOT}")
            
            # Storing Result Outputs in output_results folder
            output_folder_path = os.path.join(PROJECT_ROOT, "cmu_tare_model", "output_results")
            print(f"Result outputs will be exported here: {output_folder_path}")
            
            break  # Exit the loop if input is 'N'
        
        else:
            print("Invalid input. Please enter 'Y' or 'N'.")
    
    except Exception as e:
        print("An error occurred:", e)
        print("Please try again.")


In [ ]:
from cmu_tare_model.utils.load_exported_results_to_df import load_model_run_output

print(f"""
====================================================================================================================================================================
LOAD SCENARIO DATA
====================================================================================================================================================================
The load_model_run_output function loads scenario data from a specified folder and date. Additional details are provided below:
      
Documentation for the load_model_run_output function:
{load_model_run_output.__doc__}

-----------------------------------------------------------------------------------------------
LOADING SCENARIO DATA ...

These parameters are common to all function calls:
Output folder path: {output_folder_path}
Model run date time: {model_run_date_time}
""")

-------------------------------------------------------------------------------------------------------
# Baseline Scenario: Measure Package 0 (MP0)
-------------------------------------------------------------------------------------------------------

In [ ]:
# =======================================================================================================
# Baseline Scenario: Measure Package 0 (MP0)
# =======================================================================================================
columns_to_string = {16: str, 19: str, 20: str, 21: str}
menu_mp = 0

df_outputs_baseline_home = load_model_run_output(
    results_category='summary_baseline',
    menu_mp=menu_mp,
    output_folder_path=output_folder_path,
    location_id=location_id,
    results_export_formatted_date=model_run_date_time,
    columns_to_string=columns_to_string,
    use_chunked_loading=True,
    chunk_size=10000
)

-------------------------------------------------------------------------------------------------------
# Basic Retrofit: Measure Package 8 (MP8)
-------------------------------------------------------------------------------------------------------

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
from typing import Dict, Optional

def load_measure_package_data(
    menu_mp: int,
    output_folder_path: str,
    location_id: str,
    model_run_date_time: str,
    columns_to_string: Optional[Dict[int, type]] = None
) -> Dict[str, Dict[str, pd.DataFrame]]:
    """Load all RCM × discount rate combinations for a measure package.
    
    Creates a nested dictionary structure matching the export format:
    {rcm_model: {discount_rate_col: DataFrame}}
    
    Args:
        menu_mp: Measure package identifier (8, 9, or 10).
        output_folder_path: Base directory containing exported results.
        location_id: Geographic identifier used in filenames.
        model_run_date_time: Timestamp string from the model run.
        columns_to_string: Optional dict mapping column indices to str type.
    
    Returns:
        Nested dictionary: {rcm_model: {discount_rate_col: DataFrame}}
    """
    if columns_to_string is None:
        columns_to_string = {16: str, 19: str, 20: str, 21: str}
    
    # Initialize nested dictionary
    dataframes = {
        rcm: {dr: None for dr in PRIVATE_DISCOUNT_RATE_COLS}
        for rcm in RCM_MODELS
    }
    
    print(f"Loading MP{menu_mp} data...")
    
    for rcm_model in RCM_MODELS:
        print(f"  {rcm_model.upper()}: ", end="")
        
        for discount_rate_col in PRIVATE_DISCOUNT_RATE_COLS:
            df = load_model_run_output(
                results_category='summary',
                menu_mp=menu_mp,
                output_folder_path=output_folder_path,
                location_id=location_id,
                results_export_formatted_date=model_run_date_time,
                rcm_model=rcm_model,
                discount_rate_col=discount_rate_col,
                columns_to_string=columns_to_string,
                use_chunked_loading=True,
                chunk_size=10000
            )
            
            dataframes[rcm_model][discount_rate_col] = df
            print("✓" if df is not None else "✗", end=" ")
        
        print()  # Newline after each RCM model
    
    print(f"MP{menu_mp} loading complete!\n")
    return dataframes


def get_df(
    dataframes: Dict[str, Dict[str, pd.DataFrame]], 
    rcm: str, 
    discount: str
) -> pd.DataFrame:
    """Convenience accessor for nested dataframe dictionary.
    
    Provides shorter syntax for accessing dataframes:
        get_df(DATAFRAMES_MP8, 'inmap', 'fixed_base')
    Instead of:
        DATAFRAMES_MP8['inmap']['private_discount_rate_fixed_base']
    
    Args:
        dataframes: Nested dictionary from load_measure_package_data.
        rcm: RCM model name ('ap2', 'easiur', 'inmap').
        discount: Short discount rate name ('fixed_low', 'fixed_base', 
                  'fixed_high', 'variable').
    
    Returns:
        The requested DataFrame.
    """
    discount_key = f'private_discount_rate_{discount}'
    return dataframes[rcm][discount_key]

In [ ]:
# =======================================================================================================
# BASIC RETROFIT: MEASURE PACKAGE 8 (MP8) WITH HEALTH RCM-CRF SENSITIVITY 
# =======================================================================================================
columns_to_string = {16: str, 19: str, 20: str, 21: str}
menu_mp = 8

# Check if already loaded by checking if variable exists AND has data
if 'DATAFRAMES_MP8_RCM_DISCOUNT_RATE' in globals() and DATAFRAMES_MP8_RCM_DISCOUNT_RATE:
    print("DATAFRAMES_MP8_RCM_DISCOUNT_RATE is already loaded.")
else:
    print("Loading DATAFRAMES_MP8_RCM_DISCOUNT_RATE...")
    
    # Initialize nested dictionary structure (like your export code does)
    DATAFRAMES_MP8_RCM_DISCOUNT_RATE = {
        rcm: {dr: None for dr in PRIVATE_DISCOUNT_RATE_COLS}
        for rcm in RCM_MODELS
    }
    
    # Load each combination
    for rcm_model in RCM_MODELS:
        print(f"Loading {rcm_model.upper()} model...")
        
        for discount_rate_col in PRIVATE_DISCOUNT_RATE_COLS:
            df = load_model_run_output(
                results_category='summary',
                menu_mp=menu_mp,
                output_folder_path=output_folder_path,
                location_id=location_id,
                results_export_formatted_date=model_run_date_time,
                rcm_model=rcm_model,
                discount_rate_col=discount_rate_col,
                columns_to_string=columns_to_string,
                use_chunked_loading=True,
                chunk_size=10000
            )
            
            DATAFRAMES_MP8_RCM_DISCOUNT_RATE[rcm_model][discount_rate_col] = df
            
            if df is None:
                print(f"  Warning: Failed to load {rcm_model}/{discount_rate_col}")
        
        print()
    
    print(f"MP{menu_mp} loading complete!")

# Extract to individual variables for downstream compatibility
# ====== AP2 ======
df_outputs_mp8_ap2_FIXED_LOW = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_fixed_low']
df_outputs_mp8_ap2_FIXED_BASE = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_fixed_base']
df_outputs_mp8_ap2_FIXED_HIGH = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_fixed_high']
df_outputs_mp8_ap2_VARIABLE = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_variable']

# ====== EASIUR ======
df_outputs_mp8_easiur_FIXED_LOW = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_fixed_low']
df_outputs_mp8_easiur_FIXED_BASE = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_fixed_base']
df_outputs_mp8_easiur_FIXED_HIGH = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_fixed_high']
df_outputs_mp8_easiur_VARIABLE = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_variable']

# ====== InMAP ======
df_outputs_mp8_inmap_FIXED_LOW = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_fixed_low']
df_outputs_mp8_inmap_FIXED_BASE = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_fixed_base']
df_outputs_mp8_inmap_FIXED_HIGH = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_fixed_high']
df_outputs_mp8_inmap_VARIABLE = DATAFRAMES_MP8_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_variable']

-------------------------------------------------------------------------------------------------------
# Moderate Retrofit: Measure Package 9 (MP9)
-------------------------------------------------------------------------------------------------------

In [ ]:
# =======================================================================================================
# MODERATE RETROFIT: MEASURE PACKAGE 9 (MP9) WITH HEALTH RCM-CRF SENSITIVITY 
# =======================================================================================================
# Common parameters
columns_to_string = {16: str, 19: str, 20: str, 21: str}
menu_mp = 9

# Check if already loaded by checking if variable exists AND has data
if 'DATAFRAMES_MP9_RCM_DISCOUNT_RATE' in globals() and DATAFRAMES_MP9_RCM_DISCOUNT_RATE:
    print("DATAFRAMES_MP9_RCM_DISCOUNT_RATE is already loaded.")
else:
    print("Loading DATAFRAMES_MP9_RCM_DISCOUNT_RATE...")
    
    # Initialize nested dictionary structure (like your export code does)
    DATAFRAMES_MP9_RCM_DISCOUNT_RATE = {
        rcm: {dr: None for dr in PRIVATE_DISCOUNT_RATE_COLS}
        for rcm in RCM_MODELS
    }
    
    # Load each combination
    for rcm_model in RCM_MODELS:
        print(f"Loading {rcm_model.upper()} model...")
        
        for discount_rate_col in PRIVATE_DISCOUNT_RATE_COLS:
            df = load_model_run_output(
                results_category='summary',
                menu_mp=menu_mp,
                output_folder_path=output_folder_path,
                location_id=location_id,
                results_export_formatted_date=model_run_date_time,
                rcm_model=rcm_model,
                discount_rate_col=discount_rate_col,
                columns_to_string=columns_to_string,
                use_chunked_loading=True,
                chunk_size=10000
            )
            
            DATAFRAMES_MP9_RCM_DISCOUNT_RATE[rcm_model][discount_rate_col] = df
            
            if df is None:
                print(f"  Warning: Failed to load {rcm_model}/{discount_rate_col}")
        
        print()
    
    print(f"MP{menu_mp} loading complete!")

# Extract to individual variables for downstream compatibility
# ====== AP2 ======
df_outputs_mp9_ap2_FIXED_LOW = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_fixed_low']
df_outputs_mp9_ap2_FIXED_BASE = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_fixed_base']
df_outputs_mp9_ap2_FIXED_HIGH = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_fixed_high']
df_outputs_mp9_ap2_VARIABLE = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_variable']

# ====== EASIUR ======
df_outputs_mp9_easiur_FIXED_LOW = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_fixed_low']
df_outputs_mp9_easiur_FIXED_BASE = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_fixed_base']
df_outputs_mp9_easiur_FIXED_HIGH = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_fixed_high']
df_outputs_mp9_easiur_VARIABLE = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_variable']

# ====== InMAP ======
df_outputs_mp9_inmap_FIXED_LOW = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_fixed_low']
df_outputs_mp9_inmap_FIXED_BASE = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_fixed_base']
df_outputs_mp9_inmap_FIXED_HIGH = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_fixed_high']
df_outputs_mp9_inmap_VARIABLE = DATAFRAMES_MP9_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_variable']

-------------------------------------------------------------------------------------------------------
# Advanced Retrofit: Measure Package 10 (MP10)
-------------------------------------------------------------------------------------------------------

In [ ]:
# =======================================================================================================
# ADVANCED RETROFIT: MEASURE PACKAGE 10 (MP10) WITH HEALTH RCM-CRF SENSITIVITY 
# =======================================================================================================
# Common parameters
columns_to_string = {16: str, 19: str, 20: str, 21: str, 159: str}
menu_mp = 10

# Check if already loaded by checking if variable exists AND has data
if 'DATAFRAMES_MP10_RCM_DISCOUNT_RATE' in globals() and DATAFRAMES_MP10_RCM_DISCOUNT_RATE:
    print("DATAFRAMES_MP10_RCM_DISCOUNT_RATE is already loaded.")
else:
    print("Loading DATAFRAMES_MP10_RCM_DISCOUNT_RATE...")
    
    # Initialize nested dictionary structure (like your export code does)
    DATAFRAMES_MP10_RCM_DISCOUNT_RATE = {
        rcm: {dr: None for dr in PRIVATE_DISCOUNT_RATE_COLS}
        for rcm in RCM_MODELS
    }
    
    # Load each combination
    for rcm_model in RCM_MODELS:
        print(f"Loading {rcm_model.upper()} model...")
        
        for discount_rate_col in PRIVATE_DISCOUNT_RATE_COLS:
            df = load_model_run_output(
                results_category='summary',
                menu_mp=menu_mp,
                output_folder_path=output_folder_path,
                location_id=location_id,
                results_export_formatted_date=model_run_date_time,
                rcm_model=rcm_model,
                discount_rate_col=discount_rate_col,
                columns_to_string=columns_to_string,
                use_chunked_loading=True,
                chunk_size=10000
            )
            
            DATAFRAMES_MP10_RCM_DISCOUNT_RATE[rcm_model][discount_rate_col] = df
            
            if df is None:
                print(f"  Warning: Failed to load {rcm_model}/{discount_rate_col}")
        
        print()
    
    print(f"MP{menu_mp} loading complete!")

# Extract to individual variables for downstream compatibility
# ====== AP2 ======
df_outputs_mp10_ap2_FIXED_LOW = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_fixed_low']
df_outputs_mp10_ap2_FIXED_BASE = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_fixed_base']
df_outputs_mp10_ap2_FIXED_HIGH = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_fixed_high']
df_outputs_mp10_ap2_VARIABLE = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['ap2']['private_discount_rate_variable']

# ====== EASIUR ======
df_outputs_mp10_easiur_FIXED_LOW = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_fixed_low']
df_outputs_mp10_easiur_FIXED_BASE = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_fixed_base']
df_outputs_mp10_easiur_FIXED_HIGH = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_fixed_high']
df_outputs_mp10_easiur_VARIABLE = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['easiur']['private_discount_rate_variable']

# ====== InMAP ======
df_outputs_mp10_inmap_FIXED_LOW = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_fixed_low']
df_outputs_mp10_inmap_FIXED_BASE = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_fixed_base']
df_outputs_mp10_inmap_FIXED_HIGH = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_fixed_high']
df_outputs_mp10_inmap_VARIABLE = DATAFRAMES_MP10_RCM_DISCOUNT_RATE['inmap']['private_discount_rate_variable']

# CLIMATE CHANGE AND PUBLIC HEALTH IMPACTS

In [ ]:
from cmu_tare_model.utils.data_visualization import print_summary_stats
from cmu_tare_model.utils.data_visualization_boxplots import create_subplot_grid_boxplot
from cmu_tare_model.utils.data_visualization_histograms import create_subplot_grid_histogram, print_positive_percentages_complete

if VERBOSE:
    print(f"""  
    ====================================================================================================================================================================
    UNCERTAINTY ANALYSIS VISUALIZATION
    ====================================================================================================================================================================

    --------------------------------------------------------
    SUMMARY STATISTICS TABLE
    --------------------------------------------------------
    data_visualization.py file contains the documentation for the print_summary_stats function.

    --------------------------------------------------------
    SUBPLOT GRID OF BOXPLOTS
    --------------------------------------------------------
    data_visualization_boxplots.py file contains the documentation for the create_subplot_grid_boxplot function.
        
    --------------------------------------------------------
    SUBPLOT GRID OF HISTOGRAMS
    --------------------------------------------------------
    data_visualization_histograms.py file contains the documentation for the create_subplot_grid_histogram function.
        
    --------------------------------------------------------------------------------------------------------------------------------------------------------------------
    """)

## HEALTH IMPACT: 3 Reduced Complexity Models x 2 CR Functions

### ACS CR Function

In [ ]:
scenario_prefix = 'iraRef_mp8_'
category = 'heating'
cr_function = 'acs'
lower_percentile = 0.5
upper_percentile = 99.5

print(f"""
===== FIGURE 8.A: MONETIZED HEALTH IMPACT (HEALTH NPV, {cr_function.upper()} CR-FUNCTION) =====
- Retrofit Scenarios: {scenario_prefix}
- Categories: {category}
- RCM Models: {RCM_MODELS}
- CR Function: {cr_function}

Valid Range: {lower_percentile}th to {upper_percentile}th Percentile
""")

fig_HEATING_health_npv_ACS = create_subplot_grid_boxplot(
    dataframes=[df_outputs_mp8_ap2_FIXED_BASE, df_outputs_mp8_easiur_FIXED_BASE, df_outputs_mp8_inmap_FIXED_BASE],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    y_cols=[
        f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}',
        f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}',
        f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'
    ],
    # category_col='urbanicity',
    hue_col=f'base_{category}_fuel',
    sharex=True,
    sharey=True,
    subplot_titles=[f'AP2 ({cr_function.upper()} CR-Function)', f'EASIUR ({cr_function.upper()} CR-Function)', f'InMAP ({cr_function.upper()} CR-Function)'],
    x_labels=['', '', ''],
    y_labels=['Health NPV [2023 $USD]', '', ''],
    lower_percentile=lower_percentile,
    upper_percentile=upper_percentile,
    figure_size=(16, 6),
    show_outliers=False,
    show_xtick_labels=False  # Hide x-tick labels for cleaner look
)

print_positive_percentages_complete(
    dataframes=[
        df_outputs_mp8_ap2_FIXED_BASE,
        df_outputs_mp8_easiur_FIXED_BASE, 
        df_outputs_mp8_inmap_FIXED_BASE
    ],
    # dataframe_indices=[0, 0, 1, 1, 2, 2],  # First 2 columns from df 0, next 2 from df 1, last 2 from df 2
    column_names=[
        f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}',
        f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}',
        f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'
    ],
    subplot_titles=[
        f'AP2 with {cr_function.upper()} CR-Function',
        f'EASIUR with {cr_function.upper()} CR-Function',
        f'InMAP with {cr_function.upper()} CR-Function'
    ],
    fuel_column=f'base_{category}_fuel'
)

# Show the visual
fig_HEATING_health_npv_ACS

### H6C CR Function

In [ ]:
scenario_prefix = 'iraRef_mp8_'
category = 'heating'
cr_function = 'h6c'
lower_percentile = 0.5
upper_percentile = 99.5

print(f"""
===== FIGURE 8.A: MONETIZED HEALTH IMPACT (HEALTH NPV, {cr_function.upper()} CR-FUNCTION) =====
- Retrofit Scenarios: {scenario_prefix}
- Categories: {category}
- RCM Models: {RCM_MODELS}
- CR Function: {cr_function}

Valid Range: {lower_percentile}th to {upper_percentile}th Percentile
""")

fig_HEATING_health_npv_H6C = create_subplot_grid_boxplot(
    dataframes=[df_outputs_mp8_ap2_FIXED_BASE, df_outputs_mp8_easiur_FIXED_BASE, df_outputs_mp8_inmap_FIXED_BASE],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    y_cols=[
        f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}',
        f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}',
        f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'
    ],
    # category_col='urbanicity',
    hue_col=f'base_{category}_fuel',
    sharex=True,
    sharey=True,
    subplot_titles=[f'AP2 ({cr_function.upper()} CR-Function)', f'EASIUR ({cr_function.upper()} CR-Function)', f'InMAP ({cr_function.upper()} CR-Function)'],
    x_labels=['', '', ''],
    y_labels=['Health NPV [2023 $USD]', '', ''],
    lower_percentile=lower_percentile,
    upper_percentile=upper_percentile,
    figure_size=(16, 6),
    show_outliers=False,
    show_xtick_labels=False  # Hide x-tick labels for cleaner look
)

print_positive_percentages_complete(
    dataframes=[
        df_outputs_mp8_ap2_FIXED_BASE,
        df_outputs_mp8_easiur_FIXED_BASE, 
        df_outputs_mp8_inmap_FIXED_BASE
    ],
    # dataframe_indices=[0, 0, 1, 1, 2, 2],  # First 2 columns from df 0, next 2 from df 1, last 2 from df 2
    column_names=[
        f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}',
        f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}',
        f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'
    ],
    subplot_titles=[
        f'AP2 with {cr_function.upper()} CR-Function',
        f'EASIUR with {cr_function.upper()} CR-Function',
        f'InMAP with {cr_function.upper()} CR-Function'
    ],
    fuel_column=f'base_{category}_fuel'
)

# Show the visual
fig_HEATING_health_npv_H6C

In [ ]:
# =======================================================================================
# PRINT SUMMARY STATS: HEALTH RCM-CRF SENSITIVITY
# =======================================================================================
# ===== AP2 =====
print_summary_stats(
    df=df_outputs_mp8_ap2_FIXED_BASE,
    column_names=[
        f'{scenario_prefix}{category}_health_npv_ap2_acs',
        f'{scenario_prefix}{category}_health_npv_ap2_h6c',
        ],
    subplot_titles=[
        'AP2 with ACS CR-Function',
        'AP2 with H6C CR-Function'
        ],
    )

# ===== EASIUR =====
print_summary_stats(
    df=df_outputs_mp8_easiur_FIXED_BASE,
    column_names=[
        f'{scenario_prefix}{category}_health_npv_easiur_acs',
        f'{scenario_prefix}{category}_health_npv_easiur_h6c',
        ],
    subplot_titles=[
        'EASIUR with ACS CR-Function',
        'EASIUR with H6C CR-Function'
        ],
    )

# ===== InMAP =====
print_summary_stats(
    df=df_outputs_mp8_inmap_FIXED_BASE,
    column_names=[
        f'{scenario_prefix}{category}_health_npv_inmap_acs',
        f'{scenario_prefix}{category}_health_npv_inmap_h6c',
        ],
    subplot_titles=[
        'InMAP with ACS CR-Function',
        'InMAP with H6C CR-Function'
        ],
    )

## Climate Change Impact (SCC) and Tier 3 Adopters

### Space Heating - Progressive Impact of Climate Benefit Valuation

In [ ]:
scenario_prefix = 'iraRef_mp8_'
category = 'heating'
scc = 'central'
discount_rate = 'fixed_base'
lower_percentile = 0.5
upper_percentile = 99.5

print(f"""
===== FIGURE 7: CLIMATE BENEFIT IMPACT ON RETROFIT ADOPTION POTENTIAL (TIER 3) =====
- Retrofit Scenarios: {scenario_prefix} 
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Categories: {category}

Valid Range: {lower_percentile}th to {upper_percentile}th Percentile
""")

fig_heating_climate_scc_FIXED_BASE = create_subplot_grid_histogram(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_BASE
        ],
    subplot_positions=[(0, 0), (0, 1), (0, 2), (0, 3)],  # 1x4 grid
    x_cols=[
        f'{scenario_prefix}{category}_private_npv_moreWTP_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_lower_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_central_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_upper_{discount_rate}'
    ],
    x_labels=['Private NPV [2023 $USD]'] + ['Total NPV [2023 $USD]'] * 3,
    y_labels=['Dwelling Units', '', '', ''],
    bin_number=40,  # Optional: number of bins for histogram
    lower_percentile=lower_percentile,    # Show nearly full range
    upper_percentile=upper_percentile,   # Show nearly full range
    subplot_titles=[
        'Private NPV Only\n37% Positive NPV',
        'SCC Lower Bound\n56% Positive NPV',
        'SCC Central Estimate\n78% Positive NPV',
        'SCC Upper Bound\n83% Positive NPV' 
    ],
    # suptitle=f'{category.title()}: Progressive Impact of Climate Benefit Valuation',
    figure_size=(20, 10),  # Wide format for 4 panels
    sharex=False,  # Keep different scales to show full distributions
    sharey=True,   # Same y-scale for comparison
    color_code=f'base_{category}_fuel'
)

print_positive_percentages_complete(
    df=df_outputs_mp8_inmap_FIXED_BASE,
    column_names=[
        f'{scenario_prefix}{category}_private_npv_moreWTP_{discount_rate}',              # Private baseline
        f'{scenario_prefix}{category}_total_npv_climateOnly_lower_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_central_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_upper_{discount_rate}'
    ],
    subplot_titles=[
        f'Private NPV Only (Baseline), Discount Rate: {discount_rate}', 
        f'Lower Bound SCC (+ Climate), Discount Rate: {discount_rate}', 
        f'Central Estimate SCC (+ Climate), Discount Rate: {discount_rate}', 
        f'Upper Bound SCC (+ Climate), Discount Rate: {discount_rate}'
    ],
    fuel_column=f'base_{category}_fuel'
)

fig_heating_climate_scc_FIXED_BASE

# Adoption Rate Scenario Comparison

In [ ]:
from cmu_tare_model.adoption_potential.determine_adoption_potential_sensitivity import * 
from cmu_tare_model.adoption_potential.data_processing.visuals_adoption_potential import (
    create_multiIndex_adoption_df,
    print_adoption_decision_percentages,
    subplot_grid_adoption_vBar
)

if VERBOSE:

    print(f"""  
    ====================================================================================================================================================================
    ADOPTION POTENTIAL VISUALIZATION
    ====================================================================================================================================================================

    --------------------------------------------------------
    CREATE MULTI-INDEX DF FOR ADOPTION POTENTIAL
    --------------------------------------------------------
    visuals_adoption_potential.py file contains the documentation for the create_multiIndex_adoption_df function.

    --------------------------------------------------------
    VISUALIZE ADOPTION POTENTIAL SUBPLOT GRID
    --------------------------------------------------------
    visuals_adoption_potential.py file contains the documentation for the subplot_grid_adoption_vBar function.
        
    --------------------------------------------------------------------------------------------------------------------------------------------------------------------

    """)

## Space Heating - Basic (MP8), Moderate (MP9), Advanced (MP10) Retrofit


### FIXED 2% DISCOUNT RATE


In [ ]:
# =======================================================================================================
# SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): HEALTH RCM-CRF SENSITIVITY
# =======================================================================================================
# Common parameters
scc = 'central'
discount_rate = 'fixed_low'

# Define iteration dimensions
MEASURE_PACKAGES = [8, 9, 10]  # Basic, Moderate, Advanced

In [ ]:
print(f"""
Adoption Potential Summary Dataframes are created for the following sensitivity matrix:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Models: AP2, EASIUR, InMAP
- Health CR Functions: ACS, H6C
- Categories: Space Heating

Total combinations: {len(MEASURE_PACKAGES)} MPs × {len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs = {len(MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} DataFrames

Creating Multi-Index DataFrames...
""")

# Initialize nested dictionary to store results
# Structure: HEATING_ADOPTION_MI[mp][rcm][crf] = DataFrame
HEATING_ADOPTION_MI = {
    mp: {
        rcm: {crf: None for crf in CR_FUNCTIONS}
        for rcm in RCM_MODELS
    }
    for mp in MEASURE_PACKAGES
}

# Mapping from measure package to the loaded DataFrames
# This assumes you have DATAFRAMES_MP8_RCM_DISCOUNT_RATE, DATAFRAMES_MP9_RCM_DISCOUNT_RATE, DATAFRAMES_MP10_RCM_DISCOUNT_RATE loaded
DATAFRAMES_BY_MP = {
    8: DATAFRAMES_MP8_RCM_DISCOUNT_RATE,
    9: DATAFRAMES_MP9_RCM_DISCOUNT_RATE,
    10: DATAFRAMES_MP10_RCM_DISCOUNT_RATE
}

# Create all combinations using nested loops
for menu_mp in MEASURE_PACKAGES:
    mp_name = {8: 'Basic', 9: 'Moderate', 10: 'Advanced'}[menu_mp]
    print(f"\n{'='*80}")
    print(f"MEASURE PACKAGE {menu_mp} ({mp_name.upper()} RETROFIT)")
    print(f"{'='*80}")
    
    for rcm_model in RCM_MODELS:
        print(f"\n  RCM Model: {rcm_model.upper()}")
        
        for cr_function in CR_FUNCTIONS:
            # Get the source DataFrame from the nested dictionary
            source_df = DATAFRAMES_BY_MP[menu_mp][rcm_model][f'private_discount_rate_{discount_rate}']
            
            # Create the multi-index adoption DataFrame
            df_mi = create_multiIndex_adoption_df(
                df=source_df,
                menu_mp=menu_mp,
                category='heating',
                scc=scc,
                rcm_model=rcm_model,
                cr_function=cr_function,
                discount_rate=discount_rate
            )
            
            # Store in nested dictionary
            HEATING_ADOPTION_MI[menu_mp][rcm_model][cr_function] = df_mi
            
            print(f"    ✓ Created: MP{menu_mp} | {rcm_model.upper()} | {cr_function.upper()} | Shape: {df_mi.shape}")

print(f"\n{'='*80}")
print(f"COMPLETE: Created {len(MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} multi-index DataFrames")
print(f"{'='*80}\n")

In [ ]:
# =======================================================================================================
# EXTRACT TO INDIVIDUAL VARIABLES (for backward compatibility)
# =======================================================================================================
print(f"""-- Extracting results to dataframes for sensitivity analysis --
      SCC Bound: {scc} 
      Discount Rate: {discount_rate}""")

# ====== BASIC RETROFIT (MP8) ======
# AP2
df_mi_mp8_heating_adoption_ap2_acs_FIXED_LOW = HEATING_ADOPTION_MI[8]['ap2']['acs']
df_mi_mp8_heating_adoption_ap2_h6c_FIXED_LOW = HEATING_ADOPTION_MI[8]['ap2']['h6c']

# EASIUR
df_mi_mp8_heating_adoption_easiur_acs_FIXED_LOW = HEATING_ADOPTION_MI[8]['easiur']['acs']
df_mi_mp8_heating_adoption_easiur_h6c_FIXED_LOW = HEATING_ADOPTION_MI[8]['easiur']['h6c']

# InMAP
df_mi_mp8_heating_adoption_inmap_acs_FIXED_LOW = HEATING_ADOPTION_MI[8]['inmap']['acs']
df_mi_mp8_heating_adoption_inmap_h6c_FIXED_LOW = HEATING_ADOPTION_MI[8]['inmap']['h6c']

# ====== MODERATE RETROFIT (MP9) ======
# AP2
df_mi_mp9_heating_adoption_ap2_acs_FIXED_LOW = HEATING_ADOPTION_MI[9]['ap2']['acs']
df_mi_mp9_heating_adoption_ap2_h6c_FIXED_LOW = HEATING_ADOPTION_MI[9]['ap2']['h6c']

# EASIUR
df_mi_mp9_heating_adoption_easiur_acs_FIXED_LOW = HEATING_ADOPTION_MI[9]['easiur']['acs']
df_mi_mp9_heating_adoption_easiur_h6c_FIXED_LOW = HEATING_ADOPTION_MI[9]['easiur']['h6c']

# InMAP
df_mi_mp9_heating_adoption_inmap_acs_FIXED_LOW = HEATING_ADOPTION_MI[9]['inmap']['acs']
df_mi_mp9_heating_adoption_inmap_h6c_FIXED_LOW = HEATING_ADOPTION_MI[9]['inmap']['h6c']

# ====== ADVANCED RETROFIT (MP10) ======
# AP2
df_mi_mp10_heating_adoption_ap2_acs_FIXED_LOW = HEATING_ADOPTION_MI[10]['ap2']['acs']
df_mi_mp10_heating_adoption_ap2_h6c_FIXED_LOW = HEATING_ADOPTION_MI[10]['ap2']['h6c']

# EASIUR
df_mi_mp10_heating_adoption_easiur_acs_FIXED_LOW = HEATING_ADOPTION_MI[10]['easiur']['acs']
df_mi_mp10_heating_adoption_easiur_h6c_FIXED_LOW = HEATING_ADOPTION_MI[10]['easiur']['h6c']

# InMAP
df_mi_mp10_heating_adoption_inmap_acs_FIXED_LOW = HEATING_ADOPTION_MI[10]['inmap']['acs']
df_mi_mp10_heating_adoption_inmap_h6c_FIXED_LOW = HEATING_ADOPTION_MI[10]['inmap']['h6c']

print("✓ Extraction complete")

In [ ]:
# =======================================================================================================
# VISUALIZE SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): INMAP-ACS, FIXED-BASE DISCOUNT RATE
# =======================================================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate = 'fixed_low'

print(f"""
SENSITIVITY:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Model: {rcm_model}
- Health CR Function: {cr_function}
- Categories: Space Heating
""")

fig_all_HVAC_inmap_acs_FIXED_LOW = subplot_grid_adoption_vBar(
    dataframes=[
        df_mi_mp8_heating_adoption_inmap_acs_FIXED_LOW,
        df_mi_mp9_heating_adoption_inmap_acs_FIXED_LOW, 
        df_mi_mp10_heating_adoption_inmap_acs_FIXED_LOW
    ],
    scenarios_list=[
        [f'preIRA_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[
        "ASHP Only:\nNo IRA vs. IRA-Reference", 
        "ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference", 
        "ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference"
        ],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Space Heating Air-Source Heat Pump (ASHP) Retrofit Scenario Comparison\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

# =======================================================================================================
# PRINT ADOPTION DECISION PERCENTAGES FOR INMAP-ACS, FIXED-BASE DISCOUNT RATE
# =======================================================================================================
print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_heating_adoption_inmap_acs_FIXED_LOW,
            df_mi_mp8_heating_adoption_inmap_acs_FIXED_LOW,
            ],
        scenario_names=[
            f'preIRA_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_LOW,
            df_outputs_mp8_inmap_FIXED_LOW,
            ],
        category='heating',
        title="SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): HEALTH RCM-CRF SENSITIVITY", 
        subtitle="ASHP Only:\nNo IRA vs. IRA-Reference",
        print_header_key=True,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp9_heating_adoption_inmap_acs_FIXED_LOW, 
            df_mi_mp9_heating_adoption_inmap_acs_FIXED_LOW,
            ],
        scenario_names=[
            f'preIRA_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp9_inmap_FIXED_LOW,
            df_outputs_mp9_inmap_FIXED_LOW,
            ],
        category='heating',
        title=None,
        subtitle="ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference",
        print_header_key=False,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp10_heating_adoption_inmap_acs_FIXED_LOW, 
            df_mi_mp10_heating_adoption_inmap_acs_FIXED_LOW
            ],
        scenario_names=[
            f'preIRA_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'
            ],
        source_dataframes=[
            df_outputs_mp10_inmap_FIXED_LOW,
            df_outputs_mp10_inmap_FIXED_LOW,
            ],
        category='heating',
        title=None,
        subtitle="ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference",
        print_header_key=False,
    )

# Show the visual
fig_all_HVAC_inmap_acs_FIXED_LOW

### FIXED 7% DISCOUNT RATE


In [ ]:
# =======================================================================================================
# SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): HEALTH RCM-CRF SENSITIVITY
# =======================================================================================================
# Common parameters
scc = 'central'
discount_rate = 'fixed_base'

# Define iteration dimensions
MEASURE_PACKAGES = [8, 9, 10]  # Basic, Moderate, Advanced

In [ ]:
print(f"""
Adoption Potential Summary Dataframes are created for the following sensitivity matrix:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Models: AP2, EASIUR, InMAP
- Health CR Functions: ACS, H6C
- Categories: Space Heating

Total combinations: {len(MEASURE_PACKAGES)} MPs × {len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs = {len(MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} DataFrames

Creating Multi-Index DataFrames...
""")

# Initialize nested dictionary to store results
# Structure: HEATING_ADOPTION_MI[mp][rcm][crf] = DataFrame
HEATING_ADOPTION_MI = {
    mp: {
        rcm: {crf: None for crf in CR_FUNCTIONS}
        for rcm in RCM_MODELS
    }
    for mp in MEASURE_PACKAGES
}

# Mapping from measure package to the loaded DataFrames
# This assumes you have DATAFRAMES_MP8_RCM_DISCOUNT_RATE, DATAFRAMES_MP9_RCM_DISCOUNT_RATE, DATAFRAMES_MP10_RCM_DISCOUNT_RATE loaded
DATAFRAMES_BY_MP = {
    8: DATAFRAMES_MP8_RCM_DISCOUNT_RATE,
    9: DATAFRAMES_MP9_RCM_DISCOUNT_RATE,
    10: DATAFRAMES_MP10_RCM_DISCOUNT_RATE
}

# Create all combinations using nested loops
for menu_mp in MEASURE_PACKAGES:
    mp_name = {8: 'Basic', 9: 'Moderate', 10: 'Advanced'}[menu_mp]
    print(f"\n{'='*80}")
    print(f"MEASURE PACKAGE {menu_mp} ({mp_name.upper()} RETROFIT)")
    print(f"{'='*80}")
    
    for rcm_model in RCM_MODELS:
        print(f"\n  RCM Model: {rcm_model.upper()}")
        
        for cr_function in CR_FUNCTIONS:
            # Get the source DataFrame from the nested dictionary
            source_df = DATAFRAMES_BY_MP[menu_mp][rcm_model][f'private_discount_rate_{discount_rate}']
            
            # Create the multi-index adoption DataFrame
            df_mi = create_multiIndex_adoption_df(
                df=source_df,
                menu_mp=menu_mp,
                category='heating',
                scc=scc,
                rcm_model=rcm_model,
                cr_function=cr_function,
                discount_rate=discount_rate
            )
            
            # Store in nested dictionary
            HEATING_ADOPTION_MI[menu_mp][rcm_model][cr_function] = df_mi
            
            print(f"    ✓ Created: MP{menu_mp} | {rcm_model.upper()} | {cr_function.upper()} | Shape: {df_mi.shape}")

print(f"\n{'='*80}")
print(f"COMPLETE: Created {len(MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} multi-index DataFrames")
print(f"{'='*80}\n")

In [ ]:
# =======================================================================================================
# EXTRACT TO INDIVIDUAL VARIABLES (for backward compatibility)
# =======================================================================================================
print(f"""-- Extracting results to dataframes for sensitivity analysis --
      SCC Bound: {scc} 
      Discount Rate: {discount_rate}""")

# ====== BASIC RETROFIT (MP8) ======
# AP2
df_mi_mp8_heating_adoption_ap2_acs_FIXED_BASE = HEATING_ADOPTION_MI[8]['ap2']['acs']
df_mi_mp8_heating_adoption_ap2_h6c_FIXED_BASE = HEATING_ADOPTION_MI[8]['ap2']['h6c']

# EASIUR
df_mi_mp8_heating_adoption_easiur_acs_FIXED_BASE = HEATING_ADOPTION_MI[8]['easiur']['acs']
df_mi_mp8_heating_adoption_easiur_h6c_FIXED_BASE = HEATING_ADOPTION_MI[8]['easiur']['h6c']

# InMAP
df_mi_mp8_heating_adoption_inmap_acs_FIXED_BASE = HEATING_ADOPTION_MI[8]['inmap']['acs']
df_mi_mp8_heating_adoption_inmap_h6c_FIXED_BASE = HEATING_ADOPTION_MI[8]['inmap']['h6c']

# ====== MODERATE RETROFIT (MP9) ======
# AP2
df_mi_mp9_heating_adoption_ap2_acs_FIXED_BASE = HEATING_ADOPTION_MI[9]['ap2']['acs']
df_mi_mp9_heating_adoption_ap2_h6c_FIXED_BASE = HEATING_ADOPTION_MI[9]['ap2']['h6c']

# EASIUR
df_mi_mp9_heating_adoption_easiur_acs_FIXED_BASE = HEATING_ADOPTION_MI[9]['easiur']['acs']
df_mi_mp9_heating_adoption_easiur_h6c_FIXED_BASE = HEATING_ADOPTION_MI[9]['easiur']['h6c']

# InMAP
df_mi_mp9_heating_adoption_inmap_acs_FIXED_BASE = HEATING_ADOPTION_MI[9]['inmap']['acs']
df_mi_mp9_heating_adoption_inmap_h6c_FIXED_BASE = HEATING_ADOPTION_MI[9]['inmap']['h6c']

# ====== ADVANCED RETROFIT (MP10) ======
# AP2
df_mi_mp10_heating_adoption_ap2_acs_FIXED_BASE = HEATING_ADOPTION_MI[10]['ap2']['acs']
df_mi_mp10_heating_adoption_ap2_h6c_FIXED_BASE = HEATING_ADOPTION_MI[10]['ap2']['h6c']

# EASIUR
df_mi_mp10_heating_adoption_easiur_acs_FIXED_BASE = HEATING_ADOPTION_MI[10]['easiur']['acs']
df_mi_mp10_heating_adoption_easiur_h6c_FIXED_BASE = HEATING_ADOPTION_MI[10]['easiur']['h6c']

# InMAP
df_mi_mp10_heating_adoption_inmap_acs_FIXED_BASE = HEATING_ADOPTION_MI[10]['inmap']['acs']
df_mi_mp10_heating_adoption_inmap_h6c_FIXED_BASE = HEATING_ADOPTION_MI[10]['inmap']['h6c']

print("✓ Extraction complete")

In [ ]:
# =======================================================================================================
# VISUALIZE SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): INMAP-ACS, FIXED-BASE DISCOUNT RATE
# =======================================================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate = 'fixed_base'

print(f"""
SENSITIVITY:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Model: {rcm_model}
- Health CR Function: {cr_function}
- Categories: Space Heating
""")

fig_all_HVAC_inmap_acs_FIXED_BASE = subplot_grid_adoption_vBar(
    dataframes=[
        df_mi_mp8_heating_adoption_inmap_acs_FIXED_BASE,
        df_mi_mp9_heating_adoption_inmap_acs_FIXED_BASE, 
        df_mi_mp10_heating_adoption_inmap_acs_FIXED_BASE
    ],
    scenarios_list=[
        [f'preIRA_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[
        "ASHP Only:\nNo IRA vs. IRA-Reference", 
        "ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference", 
        "ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference"
        ],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Space Heating Air-Source Heat Pump (ASHP) Retrofit Scenario Comparison\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

# =======================================================================================================
# PRINT ADOPTION DECISION PERCENTAGES FOR INMAP-ACS, FIXED-BASE DISCOUNT RATE
# =======================================================================================================
print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_heating_adoption_inmap_acs_FIXED_BASE,
            df_mi_mp8_heating_adoption_inmap_acs_FIXED_BASE,
            ],
        scenario_names=[
            f'preIRA_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_BASE,
            df_outputs_mp8_inmap_FIXED_BASE,
            ],
        category='heating',
        title="SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): HEALTH RCM-CRF SENSITIVITY", 
        subtitle="ASHP Only:\nNo IRA vs. IRA-Reference",
        print_header_key=True,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp9_heating_adoption_inmap_acs_FIXED_BASE, 
            df_mi_mp9_heating_adoption_inmap_acs_FIXED_BASE,
            ],
        scenario_names=[
            f'preIRA_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp9_inmap_FIXED_BASE,
            df_outputs_mp9_inmap_FIXED_BASE,
            ],
        category='heating',
        title=None,
        subtitle="ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference",
        print_header_key=False,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp10_heating_adoption_inmap_acs_FIXED_BASE, 
            df_mi_mp10_heating_adoption_inmap_acs_FIXED_BASE
            ],
        scenario_names=[
            f'preIRA_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'
            ],
        source_dataframes=[
            df_outputs_mp10_inmap_FIXED_BASE,
            df_outputs_mp10_inmap_FIXED_BASE,
            ],
        category='heating',
        title=None,
        subtitle="ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference",
        print_header_key=False,
    )

# Show the visual
fig_all_HVAC_inmap_acs_FIXED_BASE

### FIXED 12% DISCOUNT RATE

In [ ]:
# =======================================================================================================
# SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): HEALTH RCM-CRF SENSITIVITY
# =======================================================================================================
# Common parameters
scc = 'central'
discount_rate = 'fixed_high'

# Define iteration dimensions
MEASURE_PACKAGES = [8, 9, 10]  # Basic, Moderate, Advanced

In [ ]:
print(f"""
Adoption Potential Summary Dataframes are created for the following sensitivity matrix:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Models: AP2, EASIUR, InMAP
- Health CR Functions: ACS, H6C
- Categories: Space Heating

Total combinations: {len(MEASURE_PACKAGES)} MPs × {len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs = {len(MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} DataFrames

Creating Multi-Index DataFrames...
""")

# Initialize nested dictionary to store results
# Structure: HEATING_ADOPTION_MI[mp][rcm][crf] = DataFrame
HEATING_ADOPTION_MI = {
    mp: {
        rcm: {crf: None for crf in CR_FUNCTIONS}
        for rcm in RCM_MODELS
    }
    for mp in MEASURE_PACKAGES
}

# Mapping from measure package to the loaded DataFrames
# This assumes you have DATAFRAMES_MP8_RCM_DISCOUNT_RATE, DATAFRAMES_MP9_RCM_DISCOUNT_RATE, DATAFRAMES_MP10_RCM_DISCOUNT_RATE loaded
DATAFRAMES_BY_MP = {
    8: DATAFRAMES_MP8_RCM_DISCOUNT_RATE,
    9: DATAFRAMES_MP9_RCM_DISCOUNT_RATE,
    10: DATAFRAMES_MP10_RCM_DISCOUNT_RATE
}

# Create all combinations using nested loops
for menu_mp in MEASURE_PACKAGES:
    mp_name = {8: 'Basic', 9: 'Moderate', 10: 'Advanced'}[menu_mp]
    print(f"\n{'='*80}")
    print(f"MEASURE PACKAGE {menu_mp} ({mp_name.upper()} RETROFIT)")
    print(f"{'='*80}")
    
    for rcm_model in RCM_MODELS:
        print(f"\n  RCM Model: {rcm_model.upper()}")
        
        for cr_function in CR_FUNCTIONS:
            # Get the source DataFrame from the nested dictionary
            source_df = DATAFRAMES_BY_MP[menu_mp][rcm_model][f'private_discount_rate_{discount_rate}']
            
            # Create the multi-index adoption DataFrame
            df_mi = create_multiIndex_adoption_df(
                df=source_df,
                menu_mp=menu_mp,
                category='heating',
                scc=scc,
                rcm_model=rcm_model,
                cr_function=cr_function,
                discount_rate=discount_rate
            )
            
            # Store in nested dictionary
            HEATING_ADOPTION_MI[menu_mp][rcm_model][cr_function] = df_mi
            
            print(f"    ✓ Created: MP{menu_mp} | {rcm_model.upper()} | {cr_function.upper()} | Shape: {df_mi.shape}")

print(f"\n{'='*80}")
print(f"COMPLETE: Created {len(MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} multi-index DataFrames")
print(f"{'='*80}\n")

In [ ]:
# =======================================================================================================
# EXTRACT TO INDIVIDUAL VARIABLES (for backward compatibility)
# =======================================================================================================
print(f"""-- Extracting results to dataframes for sensitivity analysis --
      SCC Bound: {scc} 
      Discount Rate: {discount_rate}""")

# ====== BASIC RETROFIT (MP8) ======
# AP2
df_mi_mp8_heating_adoption_ap2_acs_FIXED_HIGH = HEATING_ADOPTION_MI[8]['ap2']['acs']
df_mi_mp8_heating_adoption_ap2_h6c_FIXED_HIGH = HEATING_ADOPTION_MI[8]['ap2']['h6c']

# EASIUR
df_mi_mp8_heating_adoption_easiur_acs_FIXED_HIGH = HEATING_ADOPTION_MI[8]['easiur']['acs']
df_mi_mp8_heating_adoption_easiur_h6c_FIXED_HIGH = HEATING_ADOPTION_MI[8]['easiur']['h6c']

# InMAP
df_mi_mp8_heating_adoption_inmap_acs_FIXED_HIGH = HEATING_ADOPTION_MI[8]['inmap']['acs']
df_mi_mp8_heating_adoption_inmap_h6c_FIXED_HIGH = HEATING_ADOPTION_MI[8]['inmap']['h6c']

# ====== MODERATE RETROFIT (MP9) ======
# AP2
df_mi_mp9_heating_adoption_ap2_acs_FIXED_HIGH = HEATING_ADOPTION_MI[9]['ap2']['acs']
df_mi_mp9_heating_adoption_ap2_h6c_FIXED_HIGH = HEATING_ADOPTION_MI[9]['ap2']['h6c']

# EASIUR
df_mi_mp9_heating_adoption_easiur_acs_FIXED_HIGH = HEATING_ADOPTION_MI[9]['easiur']['acs']
df_mi_mp9_heating_adoption_easiur_h6c_FIXED_HIGH = HEATING_ADOPTION_MI[9]['easiur']['h6c']

# InMAP
df_mi_mp9_heating_adoption_inmap_acs_FIXED_HIGH = HEATING_ADOPTION_MI[9]['inmap']['acs']
df_mi_mp9_heating_adoption_inmap_h6c_FIXED_HIGH = HEATING_ADOPTION_MI[9]['inmap']['h6c']

# ====== ADVANCED RETROFIT (MP10) ======
# AP2
df_mi_mp10_heating_adoption_ap2_acs_FIXED_HIGH = HEATING_ADOPTION_MI[10]['ap2']['acs']
df_mi_mp10_heating_adoption_ap2_h6c_FIXED_HIGH = HEATING_ADOPTION_MI[10]['ap2']['h6c']

# EASIUR
df_mi_mp10_heating_adoption_easiur_acs_FIXED_HIGH = HEATING_ADOPTION_MI[10]['easiur']['acs']
df_mi_mp10_heating_adoption_easiur_h6c_FIXED_HIGH = HEATING_ADOPTION_MI[10]['easiur']['h6c']

# InMAP
df_mi_mp10_heating_adoption_inmap_acs_FIXED_HIGH = HEATING_ADOPTION_MI[10]['inmap']['acs']
df_mi_mp10_heating_adoption_inmap_h6c_FIXED_HIGH = HEATING_ADOPTION_MI[10]['inmap']['h6c']

print("✓ Extraction complete")

In [ ]:
# =======================================================================================================
# VISUALIZE SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): INMAP-ACS, FIXED-BASE DISCOUNT RATE
# =======================================================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate = 'fixed_high'

print(f"""
SENSITIVITY:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Model: {rcm_model}
- Health CR Function: {cr_function}
- Categories: Space Heating
""")

fig_all_HVAC_inmap_acs_FIXED_HIGH = subplot_grid_adoption_vBar(
    dataframes=[
        df_mi_mp8_heating_adoption_inmap_acs_FIXED_HIGH,
        df_mi_mp9_heating_adoption_inmap_acs_FIXED_HIGH, 
        df_mi_mp10_heating_adoption_inmap_acs_FIXED_HIGH
    ],
    scenarios_list=[
        [f'preIRA_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[
        "ASHP Only:\nNo IRA vs. IRA-Reference", 
        "ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference", 
        "ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference"
        ],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Space Heating Air-Source Heat Pump (ASHP) Retrofit Scenario Comparison\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

# =======================================================================================================
# PRINT ADOPTION DECISION PERCENTAGES FOR INMAP-ACS, FIXED-BASE DISCOUNT RATE
# =======================================================================================================
print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_heating_adoption_inmap_acs_FIXED_HIGH,
            df_mi_mp8_heating_adoption_inmap_acs_FIXED_HIGH,
            ],
        scenario_names=[
            f'preIRA_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_HIGH,
            df_outputs_mp8_inmap_FIXED_HIGH,
            ],
        category='heating',
        title="SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): HEALTH RCM-CRF SENSITIVITY", 
        subtitle="ASHP Only:\nNo IRA vs. IRA-Reference",
        print_header_key=True,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp9_heating_adoption_inmap_acs_FIXED_HIGH, 
            df_mi_mp9_heating_adoption_inmap_acs_FIXED_HIGH,
            ],
        scenario_names=[
            f'preIRA_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp9_inmap_FIXED_HIGH,
            df_outputs_mp9_inmap_FIXED_HIGH,
            ],
        category='heating',
        title=None,
        subtitle="ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference",
        print_header_key=False,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp10_heating_adoption_inmap_acs_FIXED_HIGH, 
            df_mi_mp10_heating_adoption_inmap_acs_FIXED_HIGH
            ],
        scenario_names=[
            f'preIRA_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'
            ],
        source_dataframes=[
            df_outputs_mp10_inmap_FIXED_HIGH,
            df_outputs_mp10_inmap_FIXED_HIGH,
            ],
        category='heating',
        title=None,
        subtitle="ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference",
        print_header_key=False,
    )

# Show the visual
fig_all_HVAC_inmap_acs_FIXED_HIGH

### VARIABLE DISCOUNT RATE

In [ ]:
scc = 'central'
discount_rate = 'variable'

print(f"""
Adoption Potential Summary Dataframes are created for the following sensitivity matrix:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Models: AP2, EASIUR, InMAP
- Health CR Functions: ACS, H6C
- Categories: Space Heating

Total combinations: {len(MEASURE_PACKAGES)} MPs × {len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs = {len(MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} DataFrames

Creating Multi-Index DataFrames...
""")

# Initialize nested dictionary to store results
# Structure: HEATING_ADOPTION_MI[mp][rcm][crf] = DataFrame
HEATING_ADOPTION_MI = {
    mp: {
        rcm: {crf: None for crf in CR_FUNCTIONS}
        for rcm in RCM_MODELS
    }
    for mp in MEASURE_PACKAGES
}

# Mapping from measure package to the loaded DataFrames
# This assumes you have DATAFRAMES_MP8_RCM_DISCOUNT_RATE, DATAFRAMES_MP9_RCM_DISCOUNT_RATE, DATAFRAMES_MP10_RCM_DISCOUNT_RATE loaded
DATAFRAMES_BY_MP = {
    8: DATAFRAMES_MP8_RCM_DISCOUNT_RATE,
    9: DATAFRAMES_MP9_RCM_DISCOUNT_RATE,
    10: DATAFRAMES_MP10_RCM_DISCOUNT_RATE
}

# Create all combinations using nested loops
for menu_mp in MEASURE_PACKAGES:
    mp_name = {8: 'Basic', 9: 'Moderate', 10: 'Advanced'}[menu_mp]
    print(f"\n{'='*80}")
    print(f"MEASURE PACKAGE {menu_mp} ({mp_name.upper()} RETROFIT)")
    print(f"{'='*80}")
    
    for rcm_model in RCM_MODELS:
        print(f"\n  RCM Model: {rcm_model.upper()}")
        
        for cr_function in CR_FUNCTIONS:
            # Get the source DataFrame from the nested dictionary
            source_df = DATAFRAMES_BY_MP[menu_mp][rcm_model][f'private_discount_rate_{discount_rate}']
            
            # Create the multi-index adoption DataFrame
            df_mi = create_multiIndex_adoption_df(
                df=source_df,
                menu_mp=menu_mp,
                category='heating',
                scc=scc,
                rcm_model=rcm_model,
                cr_function=cr_function,
                discount_rate=discount_rate
            )
            
            # Store in nested dictionary
            HEATING_ADOPTION_MI[menu_mp][rcm_model][cr_function] = df_mi
            
            print(f"    ✓ Created: MP{menu_mp} | {rcm_model.upper()} | {cr_function.upper()} | Shape: {df_mi.shape}")

print(f"\n{'='*80}")
print(f"COMPLETE: Created {len(MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} multi-index DataFrames")
print(f"{'='*80}\n")

In [ ]:
print(f"""-- Extracting results to dataframes for sensitivity analysis: --
      SCC Bound: {scc} 
      Discount Rate: {discount_rate}""")

# ====== BASIC RETROFIT (MP8) ======
# AP2
df_mi_mp8_heating_adoption_ap2_acs_VARIABLE = HEATING_ADOPTION_MI[8]['ap2']['acs']
df_mi_mp8_heating_adoption_ap2_h6c_VARIABLE = HEATING_ADOPTION_MI[8]['ap2']['h6c']

# EASIUR
df_mi_mp8_heating_adoption_easiur_acs_VARIABLE = HEATING_ADOPTION_MI[8]['easiur']['acs']
df_mi_mp8_heating_adoption_easiur_h6c_VARIABLE = HEATING_ADOPTION_MI[8]['easiur']['h6c']

# InMAP
df_mi_mp8_heating_adoption_inmap_acs_VARIABLE = HEATING_ADOPTION_MI[8]['inmap']['acs']
df_mi_mp8_heating_adoption_inmap_h6c_VARIABLE = HEATING_ADOPTION_MI[8]['inmap']['h6c']

# ====== MODERATE RETROFIT (MP9) ======
# AP2
df_mi_mp9_heating_adoption_ap2_acs_VARIABLE = HEATING_ADOPTION_MI[9]['ap2']['acs']
df_mi_mp9_heating_adoption_ap2_h6c_VARIABLE = HEATING_ADOPTION_MI[9]['ap2']['h6c']

# EASIUR
df_mi_mp9_heating_adoption_easiur_acs_VARIABLE = HEATING_ADOPTION_MI[9]['easiur']['acs']
df_mi_mp9_heating_adoption_easiur_h6c_VARIABLE = HEATING_ADOPTION_MI[9]['easiur']['h6c']

# InMAP
df_mi_mp9_heating_adoption_inmap_acs_VARIABLE = HEATING_ADOPTION_MI[9]['inmap']['acs']
df_mi_mp9_heating_adoption_inmap_h6c_VARIABLE = HEATING_ADOPTION_MI[9]['inmap']['h6c']

# ====== ADVANCED RETROFIT (MP10) ======
# AP2
df_mi_mp10_heating_adoption_ap2_acs_VARIABLE = HEATING_ADOPTION_MI[10]['ap2']['acs']
df_mi_mp10_heating_adoption_ap2_h6c_VARIABLE = HEATING_ADOPTION_MI[10]['ap2']['h6c']

# EASIUR
df_mi_mp10_heating_adoption_easiur_acs_VARIABLE = HEATING_ADOPTION_MI[10]['easiur']['acs']
df_mi_mp10_heating_adoption_easiur_h6c_VARIABLE = HEATING_ADOPTION_MI[10]['easiur']['h6c']

# InMAP
df_mi_mp10_heating_adoption_inmap_acs_VARIABLE = HEATING_ADOPTION_MI[10]['inmap']['acs']
df_mi_mp10_heating_adoption_inmap_h6c_VARIABLE = HEATING_ADOPTION_MI[10]['inmap']['h6c']

print("✓ Extraction complete")

In [ ]:
# =======================================================================================================
# VISUALIZE SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): INMAP-ACS, VARIABLE DISCOUNT RATE
# =======================================================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate = 'variable'

print(f"""
SENSITIVITY:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Model: {rcm_model}
- Health CR Function: {cr_function}
- Categories: Space Heating
""")


fig_all_HVAC_inmap_acs_VARIABLE = subplot_grid_adoption_vBar(
    dataframes=[
        df_mi_mp8_heating_adoption_inmap_acs_VARIABLE,
        df_mi_mp9_heating_adoption_inmap_acs_VARIABLE, 
        df_mi_mp10_heating_adoption_inmap_acs_VARIABLE
    ],
    scenarios_list=[
        [f'preIRA_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[
        "ASHP Only:\nNo IRA vs. IRA-Reference", 
        "ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference", 
        "ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference"
        ],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Space Heating Air-Source Heat Pump (ASHP) Retrofit Scenario Comparison\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

# =======================================================================================================
# PRINT ADOPTION DECISION PERCENTAGES FOR INMAP-ACS, VARIABLE DISCOUNT RATE
# =======================================================================================================
print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_heating_adoption_inmap_acs_VARIABLE,
            df_mi_mp8_heating_adoption_inmap_acs_VARIABLE,
            ],
        scenario_names=[
            f'preIRA_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_VARIABLE,
            df_outputs_mp8_inmap_VARIABLE,
            ],
        category='heating',
        title="SPACE HEATING ADOPTION POTENTIAL (MP8, MP9, MP10): HEALTH RCM-CRF SENSITIVITY", 
        subtitle="ASHP Only:\nNo IRA vs. IRA-Reference",
        print_header_key=True,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp9_heating_adoption_inmap_acs_VARIABLE, 
            df_mi_mp9_heating_adoption_inmap_acs_VARIABLE,
            ],
        scenario_names=[
            f'preIRA_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp9_inmap_VARIABLE,
            df_outputs_mp9_inmap_VARIABLE,
            ],
        category='heating',
        title=None,
        subtitle="ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference",
        print_header_key=False,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp10_heating_adoption_inmap_acs_VARIABLE, 
            df_mi_mp10_heating_adoption_inmap_acs_VARIABLE
            ],
        scenario_names=[
            f'preIRA_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'
            ],
        source_dataframes=[
            df_outputs_mp10_inmap_VARIABLE,
            df_outputs_mp10_inmap_VARIABLE,
            ],
        category='heating',
        title=None,
        subtitle="ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference",
        print_header_key=False,
    )

# Show the visual
fig_all_HVAC_inmap_acs_VARIABLE

## Water Heating, Clothes Drying, and Cooking - Basic Retrofit (MP8)

### FIXED LOW (2%)

In [ ]:
menu_mp = 8
scc = 'central'
discount_rate = 'fixed_low'
CATEGORIES = ['waterHeating', 'clothesDrying', 'cooking']


print(f"""
=======================================================================================================
BASIC RETROFIT: MEASURE PACKAGE {menu_mp} (MP{menu_mp}) WITH HEALTH RCM-CRF SENSITIVITY 
=======================================================================================================

Creating Multi-Index DataFrames for:
- Categories: Water Heating, Clothes Drying, Cooking
- RCM Models: {RCM_MODELS}
- CR Functions: {CR_FUNCTIONS}
- SCC: {scc}
- Discount Rate: {discount_rate}

Total combinations: {len(CATEGORIES)} categories × {len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs = {len(CATEGORIES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} DataFrames

""")

# Initialize nested dictionary to store results
# Structure: MP8_ADOPTION_MI[category][rcm][crf] = DataFrame
MP8_ADOPTION_MI = {
    category: {
        rcm: {crf: None for crf in CR_FUNCTIONS}
        for rcm in RCM_MODELS
    }
    for category in CATEGORIES
}

# Map RCM models to their source DataFrames for MP8
MP8_DATAFRAMES_BY_RCM = {
    'ap2': df_outputs_mp8_ap2_FIXED_LOW,
    'easiur': df_outputs_mp8_easiur_FIXED_LOW,
    'inmap': df_outputs_mp8_inmap_FIXED_LOW
}

# Category display names for pretty printing
CATEGORY_NAMES = {
    'waterHeating': 'Water Heating',
    'clothesDrying': 'Clothes Drying',
    'cooking': 'Cooking'
}

# Create all combinations using nested loops
for category in CATEGORIES:
    print(f"\n{'='*80}")
    print(f"CATEGORY: {CATEGORY_NAMES[category].upper()}")
    print(f"{'='*80}")
    
    for rcm_model in RCM_MODELS:
        print(f"\n  RCM Model: {rcm_model.upper()}")
        
        for cr_function in CR_FUNCTIONS:
            # Get the source DataFrame
            source_df = MP8_DATAFRAMES_BY_RCM[rcm_model]
            
            # Create the multi-index adoption DataFrame
            df_mi = create_multiIndex_adoption_df(
                df=source_df,
                menu_mp=menu_mp,
                category=category,
                scc=scc,
                rcm_model=rcm_model,
                cr_function=cr_function,
                discount_rate=discount_rate
            )
            
            # Store in nested dictionary
            MP8_ADOPTION_MI[category][rcm_model][cr_function] = df_mi
            
            print(f"    ✓ Created: {category:15} | {rcm_model.upper():6} | {cr_function.upper():3} | Shape: {df_mi.shape}")

print(f"\n{'='*80}")
print(f"COMPLETE: Created {len(CATEGORIES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} multi-index DataFrames for MP{menu_mp}")
print(f"{'='*80}\n")

In [ ]:
# =======================================================================================================
# EXTRACT TO INDIVIDUAL VARIABLES FOR ADOPTION POTENTIAL SUBPLOT GRID VISUALS
# =======================================================================================================
print("Extracting to individual df variables for subplot grid visuals ...")

# Water Heating
df_mi_mp8_waterHeating_adoption_ap2_acs_FIXED_LOW = MP8_ADOPTION_MI['waterHeating']['ap2']['acs']
df_mi_mp8_waterHeating_adoption_ap2_h6c_FIXED_LOW = MP8_ADOPTION_MI['waterHeating']['ap2']['h6c']
df_mi_mp8_waterHeating_adoption_easiur_acs_FIXED_LOW = MP8_ADOPTION_MI['waterHeating']['easiur']['acs']
df_mi_mp8_waterHeating_adoption_easiur_h6c_FIXED_LOW = MP8_ADOPTION_MI['waterHeating']['easiur']['h6c']
df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_LOW = MP8_ADOPTION_MI['waterHeating']['inmap']['acs']
df_mi_mp8_waterHeating_adoption_inmap_h6c_FIXED_LOW = MP8_ADOPTION_MI['waterHeating']['inmap']['h6c']

# Clothes Drying
df_mi_mp8_clothesDrying_adoption_ap2_acs_FIXED_LOW = MP8_ADOPTION_MI['clothesDrying']['ap2']['acs']
df_mi_mp8_clothesDrying_adoption_ap2_h6c_FIXED_LOW = MP8_ADOPTION_MI['clothesDrying']['ap2']['h6c']
df_mi_mp8_clothesDrying_adoption_easiur_acs_FIXED_LOW = MP8_ADOPTION_MI['clothesDrying']['easiur']['acs']
df_mi_mp8_clothesDrying_adoption_easiur_h6c_FIXED_LOW = MP8_ADOPTION_MI['clothesDrying']['easiur']['h6c']
df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_LOW = MP8_ADOPTION_MI['clothesDrying']['inmap']['acs']
df_mi_mp8_clothesDrying_adoption_inmap_h6c_FIXED_LOW = MP8_ADOPTION_MI['clothesDrying']['inmap']['h6c']

# Cooking
df_mi_mp8_cooking_adoption_ap2_acs_FIXED_LOW = MP8_ADOPTION_MI['cooking']['ap2']['acs']
df_mi_mp8_cooking_adoption_ap2_h6c_FIXED_LOW = MP8_ADOPTION_MI['cooking']['ap2']['h6c']
df_mi_mp8_cooking_adoption_easiur_acs_FIXED_LOW = MP8_ADOPTION_MI['cooking']['easiur']['acs']
df_mi_mp8_cooking_adoption_easiur_h6c_FIXED_LOW = MP8_ADOPTION_MI['cooking']['easiur']['h6c']
df_mi_mp8_cooking_adoption_inmap_acs_FIXED_LOW = MP8_ADOPTION_MI['cooking']['inmap']['acs']
df_mi_mp8_cooking_adoption_inmap_h6c_FIXED_LOW = MP8_ADOPTION_MI['cooking']['inmap']['h6c']

print("✓ Extraction complete")

In [ ]:
# I used this function call and the visual is still displaying income_level.

# ====================================================================
# 1. EQUIPMENT COMPARISON: Water Heating, Clothes Drying, Cooking - Basic Retrofit (MP8)
# ====================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate = 'fixed_low'

print(f"""
SENSITIVITY:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Model: {rcm_model}
- Health CR Function: {cr_function}
- Categories: Space Heating
""")

# Assign to a variable to prevent duplicate display
fig_mp8_nonHVAC_inmap_acs_FIXED_LOW = subplot_grid_adoption_vBar(
    dataframes=[
        df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_LOW,
        df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_LOW, 
        df_mi_mp8_cooking_adoption_inmap_acs_FIXED_LOW
    ],
    scenarios_list=[
        [f'preIRA_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[
        "Heat Pump Water Heater:\nNo IRA vs. IRA-Reference",
        "Heat Pump Clothes Dryer:\nNo IRA vs. IRA-Reference",
        "Electric Resistance Range:\nNo IRA vs. IRA-Reference"
    ],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Basic Retrofit (MP8): High-Efficiency Equipment Electrification\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_LOW, df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_LOW,
            ],
        scenario_names=[
            f'preIRA_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_LOW,
            df_outputs_mp8_inmap_FIXED_LOW,
            ],
        category='waterHeating',
        title="EQUIPMENT COMPARISON: Water Heating, Clothes Drying, Cooking - Basic Retrofit (MP8)",
        subtitle="Heat Pump Water Heater: Central SCC|InMAP|ACS",
        print_header_key=True,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_LOW, df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_LOW,
            ],
        scenario_names=[
            f'preIRA_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_LOW,
            df_outputs_mp8_inmap_FIXED_LOW,
            ],
        category='clothesDrying',
        title=None,
        subtitle="Heat Pump Clothes Dryer: Central SCC|InMAP|ACS",
        print_header_key=False,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_cooking_adoption_inmap_acs_FIXED_LOW, df_mi_mp8_cooking_adoption_inmap_acs_FIXED_LOW
            ],
        scenario_names=[
            f'preIRA_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_LOW,
            df_outputs_mp8_inmap_FIXED_LOW,
            ],
        category='cooking',
        title=None,
        subtitle="Electric Resistance Range: Central SCC|InMAP|ACS",
        print_header_key=False,
    )

fig_mp8_nonHVAC_inmap_acs_FIXED_LOW

### FIXED BASE (7%)

In [ ]:
menu_mp = 8
scc = 'central'
discount_rate = 'fixed_base'
CATEGORIES = ['waterHeating', 'clothesDrying', 'cooking']


print(f"""
=======================================================================================================
BASIC RETROFIT: MEASURE PACKAGE {menu_mp} (MP{menu_mp}) WITH HEALTH RCM-CRF SENSITIVITY 
=======================================================================================================

Creating Multi-Index DataFrames for:
- Categories: Water Heating, Clothes Drying, Cooking
- RCM Models: {RCM_MODELS}
- CR Functions: {CR_FUNCTIONS}
- SCC: {scc}
- Discount Rate: {discount_rate}

Total combinations: {len(CATEGORIES)} categories × {len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs = {len(CATEGORIES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} DataFrames

""")

# Initialize nested dictionary to store results
# Structure: MP8_ADOPTION_MI[category][rcm][crf] = DataFrame
MP8_ADOPTION_MI = {
    category: {
        rcm: {crf: None for crf in CR_FUNCTIONS}
        for rcm in RCM_MODELS
    }
    for category in CATEGORIES
}

# Map RCM models to their source DataFrames for MP8
MP8_DATAFRAMES_BY_RCM = {
    'ap2': df_outputs_mp8_ap2_FIXED_BASE,
    'easiur': df_outputs_mp8_easiur_FIXED_BASE,
    'inmap': df_outputs_mp8_inmap_FIXED_BASE
}

# Category display names for pretty printing
CATEGORY_NAMES = {
    'waterHeating': 'Water Heating',
    'clothesDrying': 'Clothes Drying',
    'cooking': 'Cooking'
}

# Create all combinations using nested loops
for category in CATEGORIES:
    print(f"\n{'='*80}")
    print(f"CATEGORY: {CATEGORY_NAMES[category].upper()}")
    print(f"{'='*80}")
    
    for rcm_model in RCM_MODELS:
        print(f"\n  RCM Model: {rcm_model.upper()}")
        
        for cr_function in CR_FUNCTIONS:
            # Get the source DataFrame
            source_df = MP8_DATAFRAMES_BY_RCM[rcm_model]
            
            # Create the multi-index adoption DataFrame
            df_mi = create_multiIndex_adoption_df(
                df=source_df,
                menu_mp=menu_mp,
                category=category,
                scc=scc,
                rcm_model=rcm_model,
                cr_function=cr_function,
                discount_rate=discount_rate
            )
            
            # Store in nested dictionary
            MP8_ADOPTION_MI[category][rcm_model][cr_function] = df_mi
            
            print(f"    ✓ Created: {category:15} | {rcm_model.upper():6} | {cr_function.upper():3} | Shape: {df_mi.shape}")

print(f"\n{'='*80}")
print(f"COMPLETE: Created {len(CATEGORIES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} multi-index DataFrames for MP{menu_mp}")
print(f"{'='*80}\n")

In [ ]:
# =======================================================================================================
# EXTRACT TO INDIVIDUAL VARIABLES FOR ADOPTION POTENTIAL SUBPLOT GRID VISUALS
# =======================================================================================================
print("Extracting to individual df variables for subplot grid visuals ...")

# Water Heating
df_mi_mp8_waterHeating_adoption_ap2_acs_FIXED_BASE = MP8_ADOPTION_MI['waterHeating']['ap2']['acs']
df_mi_mp8_waterHeating_adoption_ap2_h6c_FIXED_BASE = MP8_ADOPTION_MI['waterHeating']['ap2']['h6c']
df_mi_mp8_waterHeating_adoption_easiur_acs_FIXED_BASE = MP8_ADOPTION_MI['waterHeating']['easiur']['acs']
df_mi_mp8_waterHeating_adoption_easiur_h6c_FIXED_BASE = MP8_ADOPTION_MI['waterHeating']['easiur']['h6c']
df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_BASE = MP8_ADOPTION_MI['waterHeating']['inmap']['acs']
df_mi_mp8_waterHeating_adoption_inmap_h6c_FIXED_BASE = MP8_ADOPTION_MI['waterHeating']['inmap']['h6c']

# Clothes Drying
df_mi_mp8_clothesDrying_adoption_ap2_acs_FIXED_BASE = MP8_ADOPTION_MI['clothesDrying']['ap2']['acs']
df_mi_mp8_clothesDrying_adoption_ap2_h6c_FIXED_BASE = MP8_ADOPTION_MI['clothesDrying']['ap2']['h6c']
df_mi_mp8_clothesDrying_adoption_easiur_acs_FIXED_BASE = MP8_ADOPTION_MI['clothesDrying']['easiur']['acs']
df_mi_mp8_clothesDrying_adoption_easiur_h6c_FIXED_BASE = MP8_ADOPTION_MI['clothesDrying']['easiur']['h6c']
df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_BASE = MP8_ADOPTION_MI['clothesDrying']['inmap']['acs']
df_mi_mp8_clothesDrying_adoption_inmap_h6c_FIXED_BASE = MP8_ADOPTION_MI['clothesDrying']['inmap']['h6c']

# Cooking
df_mi_mp8_cooking_adoption_ap2_acs_FIXED_BASE = MP8_ADOPTION_MI['cooking']['ap2']['acs']
df_mi_mp8_cooking_adoption_ap2_h6c_FIXED_BASE = MP8_ADOPTION_MI['cooking']['ap2']['h6c']
df_mi_mp8_cooking_adoption_easiur_acs_FIXED_BASE = MP8_ADOPTION_MI['cooking']['easiur']['acs']
df_mi_mp8_cooking_adoption_easiur_h6c_FIXED_BASE = MP8_ADOPTION_MI['cooking']['easiur']['h6c']
df_mi_mp8_cooking_adoption_inmap_acs_FIXED_BASE = MP8_ADOPTION_MI['cooking']['inmap']['acs']
df_mi_mp8_cooking_adoption_inmap_h6c_FIXED_BASE = MP8_ADOPTION_MI['cooking']['inmap']['h6c']

print("✓ Extraction complete")

In [ ]:
# I used this function call and the visual is still displaying income_level.

# ====================================================================
# 1. EQUIPMENT COMPARISON: Water Heating, Clothes Drying, Cooking - Basic Retrofit (MP8)
# ====================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate = 'fixed_base'

print(f"""
SENSITIVITY:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Model: {rcm_model}
- Health CR Function: {cr_function}
- Categories: Space Heating
""")

# Assign to a variable to prevent duplicate display
fig_mp8_nonHVAC_inmap_acs_FIXED_BASE = subplot_grid_adoption_vBar(
    dataframes=[
        df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_BASE,
        df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_BASE, 
        df_mi_mp8_cooking_adoption_inmap_acs_FIXED_BASE
    ],
    scenarios_list=[
        [f'preIRA_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[
        "Heat Pump Water Heater:\nNo IRA vs. IRA-Reference",
        "Heat Pump Clothes Dryer:\nNo IRA vs. IRA-Reference",
        "Electric Resistance Range:\nNo IRA vs. IRA-Reference"
    ],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Basic Retrofit (MP8): High-Efficiency Equipment Electrification\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_BASE, df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_BASE,
            ],
        scenario_names=[
            f'preIRA_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_BASE,
            df_outputs_mp8_inmap_FIXED_BASE,
            ],
        category='waterHeating',
        title="EQUIPMENT COMPARISON: Water Heating, Clothes Drying, Cooking - Basic Retrofit (MP8)",
        subtitle="Heat Pump Water Heater: Central SCC|InMAP|ACS",
        print_header_key=True,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_BASE, df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_BASE,
            ],
        scenario_names=[
            f'preIRA_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_BASE,
            df_outputs_mp8_inmap_FIXED_BASE,
            ],
        category='clothesDrying',
        title=None,
        subtitle="Heat Pump Clothes Dryer: Central SCC|InMAP|ACS",
        print_header_key=False,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_cooking_adoption_inmap_acs_FIXED_BASE, df_mi_mp8_cooking_adoption_inmap_acs_FIXED_BASE
            ],
        scenario_names=[
            f'preIRA_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_BASE,
            df_outputs_mp8_inmap_FIXED_BASE,
            ],
        category='cooking',
        title=None,
        subtitle="Electric Resistance Range: Central SCC|InMAP|ACS",
        print_header_key=False,
    )

fig_mp8_nonHVAC_inmap_acs_FIXED_BASE

### FIXED HIGH (12%)

In [ ]:
menu_mp = 8
scc = 'central'
discount_rate = 'fixed_high'
CATEGORIES = ['waterHeating', 'clothesDrying', 'cooking']


print(f"""
=======================================================================================================
BASIC RETROFIT: MEASURE PACKAGE {menu_mp} (MP{menu_mp}) WITH HEALTH RCM-CRF SENSITIVITY 
=======================================================================================================

Creating Multi-Index DataFrames for:
- Categories: Water Heating, Clothes Drying, Cooking
- RCM Models: {RCM_MODELS}
- CR Functions: {CR_FUNCTIONS}
- SCC: {scc}
- Discount Rate: {discount_rate}

Total combinations: {len(CATEGORIES)} categories × {len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs = {len(CATEGORIES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} DataFrames

""")

# Initialize nested dictionary to store results
# Structure: MP8_ADOPTION_MI[category][rcm][crf] = DataFrame
MP8_ADOPTION_MI = {
    category: {
        rcm: {crf: None for crf in CR_FUNCTIONS}
        for rcm in RCM_MODELS
    }
    for category in CATEGORIES
}

# Map RCM models to their source DataFrames for MP8
MP8_DATAFRAMES_BY_RCM = {
    'ap2': df_outputs_mp8_ap2_FIXED_HIGH,
    'easiur': df_outputs_mp8_easiur_FIXED_HIGH,
    'inmap': df_outputs_mp8_inmap_FIXED_HIGH
}

# Category display names for pretty printing
CATEGORY_NAMES = {
    'waterHeating': 'Water Heating',
    'clothesDrying': 'Clothes Drying',
    'cooking': 'Cooking'
}

# Create all combinations using nested loops
for category in CATEGORIES:
    print(f"\n{'='*80}")
    print(f"CATEGORY: {CATEGORY_NAMES[category].upper()}")
    print(f"{'='*80}")
    
    for rcm_model in RCM_MODELS:
        print(f"\n  RCM Model: {rcm_model.upper()}")
        
        for cr_function in CR_FUNCTIONS:
            # Get the source DataFrame
            source_df = MP8_DATAFRAMES_BY_RCM[rcm_model]
            
            # Create the multi-index adoption DataFrame
            df_mi = create_multiIndex_adoption_df(
                df=source_df,
                menu_mp=menu_mp,
                category=category,
                scc=scc,
                rcm_model=rcm_model,
                cr_function=cr_function,
                discount_rate=discount_rate
            )
            
            # Store in nested dictionary
            MP8_ADOPTION_MI[category][rcm_model][cr_function] = df_mi
            
            print(f"    ✓ Created: {category:15} | {rcm_model.upper():6} | {cr_function.upper():3} | Shape: {df_mi.shape}")

print(f"\n{'='*80}")
print(f"COMPLETE: Created {len(CATEGORIES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} multi-index DataFrames for MP{menu_mp} with discount rate '{discount_rate}' and SCC '{scc}'")
print(f"{'='*80}\n")

In [ ]:
# =======================================================================================================
# EXTRACT TO INDIVIDUAL VARIABLES FOR ADOPTION POTENTIAL SUBPLOT GRID VISUALS
# =======================================================================================================
print("Extracting to individual df variables for subplot grid visuals ...")

# Water Heating
df_mi_mp8_waterHeating_adoption_ap2_acs_FIXED_HIGH = MP8_ADOPTION_MI['waterHeating']['ap2']['acs']
df_mi_mp8_waterHeating_adoption_ap2_h6c_FIXED_HIGH = MP8_ADOPTION_MI['waterHeating']['ap2']['h6c']
df_mi_mp8_waterHeating_adoption_easiur_acs_FIXED_HIGH = MP8_ADOPTION_MI['waterHeating']['easiur']['acs']
df_mi_mp8_waterHeating_adoption_easiur_h6c_FIXED_HIGH = MP8_ADOPTION_MI['waterHeating']['easiur']['h6c']
df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_HIGH = MP8_ADOPTION_MI['waterHeating']['inmap']['acs']
df_mi_mp8_waterHeating_adoption_inmap_h6c_FIXED_HIGH = MP8_ADOPTION_MI['waterHeating']['inmap']['h6c']

# Clothes Drying
df_mi_mp8_clothesDrying_adoption_ap2_acs_FIXED_HIGH = MP8_ADOPTION_MI['clothesDrying']['ap2']['acs']
df_mi_mp8_clothesDrying_adoption_ap2_h6c_FIXED_HIGH = MP8_ADOPTION_MI['clothesDrying']['ap2']['h6c']
df_mi_mp8_clothesDrying_adoption_easiur_acs_FIXED_HIGH = MP8_ADOPTION_MI['clothesDrying']['easiur']['acs']
df_mi_mp8_clothesDrying_adoption_easiur_h6c_FIXED_HIGH = MP8_ADOPTION_MI['clothesDrying']['easiur']['h6c']
df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_HIGH = MP8_ADOPTION_MI['clothesDrying']['inmap']['acs']
df_mi_mp8_clothesDrying_adoption_inmap_h6c_FIXED_HIGH = MP8_ADOPTION_MI['clothesDrying']['inmap']['h6c']

# Cooking
df_mi_mp8_cooking_adoption_ap2_acs_FIXED_HIGH = MP8_ADOPTION_MI['cooking']['ap2']['acs']
df_mi_mp8_cooking_adoption_ap2_h6c_FIXED_HIGH = MP8_ADOPTION_MI['cooking']['ap2']['h6c']
df_mi_mp8_cooking_adoption_easiur_acs_FIXED_HIGH = MP8_ADOPTION_MI['cooking']['easiur']['acs']
df_mi_mp8_cooking_adoption_easiur_h6c_FIXED_HIGH = MP8_ADOPTION_MI['cooking']['easiur']['h6c']
df_mi_mp8_cooking_adoption_inmap_acs_FIXED_HIGH = MP8_ADOPTION_MI['cooking']['inmap']['acs']
df_mi_mp8_cooking_adoption_inmap_h6c_FIXED_HIGH = MP8_ADOPTION_MI['cooking']['inmap']['h6c']

print("✓ Extraction complete")

In [ ]:
# I used this function call and the visual is still displaying income_level.

# ====================================================================
# 1. EQUIPMENT COMPARISON: Water Heating, Clothes Drying, Cooking - Basic Retrofit (MP8)
# ====================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate = 'fixed_high'

print(f"""
SENSITIVITY:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Model: {rcm_model}
- Health CR Function: {cr_function}
- Categories: Space Heating
""")

# Assign to a variable to prevent duplicate display
fig_mp8_nonHVAC_inmap_acs_FIXED_HIGH = subplot_grid_adoption_vBar(
    dataframes=[
        df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_HIGH,
        df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_HIGH, 
        df_mi_mp8_cooking_adoption_inmap_acs_FIXED_HIGH
    ],
    scenarios_list=[
        [f'preIRA_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[
        "Heat Pump Water Heater:\nNo IRA vs. IRA-Reference",
        "Heat Pump Clothes Dryer:\nNo IRA vs. IRA-Reference",
        "Electric Resistance Range:\nNo IRA vs. IRA-Reference"
    ],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Basic Retrofit (MP8): High-Efficiency Equipment Electrification\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_HIGH, df_mi_mp8_waterHeating_adoption_inmap_acs_FIXED_HIGH,
            ],
        scenario_names=[
            f'preIRA_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_HIGH,
            df_outputs_mp8_inmap_FIXED_HIGH,
            ],
        category='waterHeating',
        title="EQUIPMENT COMPARISON: Water Heating, Clothes Drying, Cooking - Basic Retrofit (MP8)",
        subtitle="Heat Pump Water Heater: Central SCC|InMAP|ACS",
        print_header_key=True,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_HIGH, df_mi_mp8_clothesDrying_adoption_inmap_acs_FIXED_HIGH,
            ],
        scenario_names=[
            f'preIRA_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_HIGH,
            df_outputs_mp8_inmap_FIXED_HIGH,
            ],
        category='clothesDrying',
        title=None,
        subtitle="Heat Pump Clothes Dryer: Central SCC|InMAP|ACS",
        print_header_key=False,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_cooking_adoption_inmap_acs_FIXED_HIGH, df_mi_mp8_cooking_adoption_inmap_acs_FIXED_HIGH
            ],
        scenario_names=[
            f'preIRA_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_FIXED_HIGH,
            df_outputs_mp8_inmap_FIXED_HIGH,
            ],
        category='cooking',
        title=None,
        subtitle="Electric Resistance Range: Central SCC|InMAP|ACS",
        print_header_key=False,
    )

fig_mp8_nonHVAC_inmap_acs_FIXED_HIGH

### VARIABLE (7-45%)

In [ ]:
menu_mp = 8
scc = 'central'
discount_rate = 'variable'
CATEGORIES = ['waterHeating', 'clothesDrying', 'cooking']


print(f"""
=======================================================================================================
BASIC RETROFIT: MEASURE PACKAGE {menu_mp} (MP{menu_mp}) WITH HEALTH RCM-CRF SENSITIVITY 
=======================================================================================================

Creating Multi-Index DataFrames for:
- Categories: Water Heating, Clothes Drying, Cooking
- RCM Models: {RCM_MODELS}
- CR Functions: {CR_FUNCTIONS}
- SCC: {scc}
- Discount Rate: {discount_rate}

Total combinations: {len(CATEGORIES)} categories × {len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs = {len(CATEGORIES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} DataFrames

""")

# Initialize nested dictionary to store results
# Structure: MP8_ADOPTION_MI[category][rcm][crf] = DataFrame
MP8_ADOPTION_MI = {
    category: {
        rcm: {crf: None for crf in CR_FUNCTIONS}
        for rcm in RCM_MODELS
    }
    for category in CATEGORIES
}

# Map RCM models to their source DataFrames for MP8
MP8_DATAFRAMES_BY_RCM = {
    'ap2': df_outputs_mp8_ap2_VARIABLE,
    'easiur': df_outputs_mp8_easiur_VARIABLE,
    'inmap': df_outputs_mp8_inmap_VARIABLE
}

# Category display names for pretty printing
CATEGORY_NAMES = {
    'waterHeating': 'Water Heating',
    'clothesDrying': 'Clothes Drying',
    'cooking': 'Cooking'
}

# Create all combinations using nested loops
for category in CATEGORIES:
    print(f"\n{'='*80}")
    print(f"CATEGORY: {CATEGORY_NAMES[category].upper()}")
    print(f"{'='*80}")
    
    for rcm_model in RCM_MODELS:
        print(f"\n  RCM Model: {rcm_model.upper()}")
        
        for cr_function in CR_FUNCTIONS:
            # Get the source DataFrame
            source_df = MP8_DATAFRAMES_BY_RCM[rcm_model]
            
            # Create the multi-index adoption DataFrame
            df_mi = create_multiIndex_adoption_df(
                df=source_df,
                menu_mp=menu_mp,
                category=category,
                scc=scc,
                rcm_model=rcm_model,
                cr_function=cr_function,
                discount_rate=discount_rate
            )
            
            # Store in nested dictionary
            MP8_ADOPTION_MI[category][rcm_model][cr_function] = df_mi
            
            print(f"    ✓ Created: {category:15} | {rcm_model.upper():6} | {cr_function.upper():3} | Shape: {df_mi.shape}")

print(f"\n{'='*80}")
print(f"COMPLETE: Created {len(CATEGORIES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} multi-index DataFrames for MP{menu_mp}")
print(f"{'='*80}\n")

In [ ]:
# =======================================================================================================
# EXTRACT TO INDIVIDUAL VARIABLES FOR ADOPTION POTENTIAL SUBPLOT GRID VISUALS
# =======================================================================================================
print("Extracting to individual df variables for subplot grid visuals ...")

# Water Heating
df_mi_mp8_waterHeating_adoption_ap2_acs_VARIABLE = MP8_ADOPTION_MI['waterHeating']['ap2']['acs']
df_mi_mp8_waterHeating_adoption_ap2_h6c_VARIABLE = MP8_ADOPTION_MI['waterHeating']['ap2']['h6c']
df_mi_mp8_waterHeating_adoption_easiur_acs_VARIABLE = MP8_ADOPTION_MI['waterHeating']['easiur']['acs']
df_mi_mp8_waterHeating_adoption_easiur_h6c_VARIABLE = MP8_ADOPTION_MI['waterHeating']['easiur']['h6c']
df_mi_mp8_waterHeating_adoption_inmap_acs_VARIABLE = MP8_ADOPTION_MI['waterHeating']['inmap']['acs']
df_mi_mp8_waterHeating_adoption_inmap_h6c_VARIABLE = MP8_ADOPTION_MI['waterHeating']['inmap']['h6c']

# Clothes Drying
df_mi_mp8_clothesDrying_adoption_ap2_acs_VARIABLE = MP8_ADOPTION_MI['clothesDrying']['ap2']['acs']
df_mi_mp8_clothesDrying_adoption_ap2_h6c_VARIABLE = MP8_ADOPTION_MI['clothesDrying']['ap2']['h6c']
df_mi_mp8_clothesDrying_adoption_easiur_acs_VARIABLE = MP8_ADOPTION_MI['clothesDrying']['easiur']['acs']
df_mi_mp8_clothesDrying_adoption_easiur_h6c_VARIABLE = MP8_ADOPTION_MI['clothesDrying']['easiur']['h6c']
df_mi_mp8_clothesDrying_adoption_inmap_acs_VARIABLE = MP8_ADOPTION_MI['clothesDrying']['inmap']['acs']
df_mi_mp8_clothesDrying_adoption_inmap_h6c_VARIABLE = MP8_ADOPTION_MI['clothesDrying']['inmap']['h6c']

# Cooking
df_mi_mp8_cooking_adoption_ap2_acs_VARIABLE = MP8_ADOPTION_MI['cooking']['ap2']['acs']
df_mi_mp8_cooking_adoption_ap2_h6c_VARIABLE = MP8_ADOPTION_MI['cooking']['ap2']['h6c']
df_mi_mp8_cooking_adoption_easiur_acs_VARIABLE = MP8_ADOPTION_MI['cooking']['easiur']['acs']
df_mi_mp8_cooking_adoption_easiur_h6c_VARIABLE = MP8_ADOPTION_MI['cooking']['easiur']['h6c']
df_mi_mp8_cooking_adoption_inmap_acs_VARIABLE = MP8_ADOPTION_MI['cooking']['inmap']['acs']
df_mi_mp8_cooking_adoption_inmap_h6c_VARIABLE = MP8_ADOPTION_MI['cooking']['inmap']['h6c']

print("✓ Extraction complete")

In [ ]:
# I used this function call and the visual is still displaying income_level.

# ====================================================================
# 1. EQUIPMENT COMPARISON: Water Heating, Clothes Drying, Cooking - Basic Retrofit (MP8)
# ====================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate = 'variable'

print(f"""
SENSITIVITY:
- Retrofit Scenarios: Basic (MP8), Moderate (MP9), Advanced (MP10)
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Health RCM Model: {rcm_model}
- Health CR Function: {cr_function}
- Categories: Space Heating
""")

# Assign to a variable to prevent duplicate display
fig_mp8_nonHVAC_inmap_acs_VARIABLE = subplot_grid_adoption_vBar(
    dataframes=[
        df_mi_mp8_waterHeating_adoption_inmap_acs_VARIABLE,
        df_mi_mp8_clothesDrying_adoption_inmap_acs_VARIABLE, 
        df_mi_mp8_cooking_adoption_inmap_acs_VARIABLE
    ],
    scenarios_list=[
        [f'preIRA_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[
        "Heat Pump Water Heater:\nNo IRA vs. IRA-Reference",
        "Heat Pump Clothes Dryer:\nNo IRA vs. IRA-Reference",
        "Electric Resistance Range:\nNo IRA vs. IRA-Reference"
    ],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Basic Retrofit (MP8): High-Efficiency Equipment Electrification\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_waterHeating_adoption_inmap_acs_VARIABLE, df_mi_mp8_waterHeating_adoption_inmap_acs_VARIABLE,
            ],
        scenario_names=[
            f'preIRA_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_VARIABLE,
            df_outputs_mp8_inmap_VARIABLE,
            ],
        category='waterHeating',
        title="EQUIPMENT COMPARISON: Water Heating, Clothes Drying, Cooking - Basic Retrofit (MP8)",
        subtitle="Heat Pump Water Heater: Central SCC|InMAP|ACS",
        print_header_key=True,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_clothesDrying_adoption_inmap_acs_VARIABLE, df_mi_mp8_clothesDrying_adoption_inmap_acs_VARIABLE,
            ],
        scenario_names=[
            f'preIRA_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_VARIABLE,
            df_outputs_mp8_inmap_VARIABLE,
            ],
        category='clothesDrying',
        title=None,
        subtitle="Heat Pump Clothes Dryer: Central SCC|InMAP|ACS",
        print_header_key=False,
    )

print_adoption_decision_percentages(
        dataframes=[
            df_mi_mp8_cooking_adoption_inmap_acs_VARIABLE, df_mi_mp8_cooking_adoption_inmap_acs_VARIABLE
            ],
        scenario_names=[
            f'preIRA_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
            f'iraRef_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'
            ],
        source_dataframes=[
            df_outputs_mp8_inmap_VARIABLE,
            df_outputs_mp8_inmap_VARIABLE,
            ],
        category='cooking',
        title=None,
        subtitle="Electric Resistance Range: Central SCC|InMAP|ACS",
        print_header_key=False,
    )

fig_mp8_nonHVAC_inmap_acs_VARIABLE

# SENSITIVITY ANALYSIS: Private Discount Rate and Adoption Feasibility (Retrofit Lifecycle Cost)

In [ ]:
# Discount Rate Sensitivity Analysis
category = 'heating'

fig_HEATING_preIRA_private_more_WTP_discount = create_subplot_grid_histogram(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    # dataframe_indices=[0, 0],
    subplot_positions=[(0, 0), (0, 1), (0, 2), (0, 3)],
    x_cols=[
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_low',
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_base',
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_high',
        f'preIRA_mp8_{category}_private_npv_moreWTP_variable'
    ],
    x_labels=['Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]'
    ],
    y_labels=['Dwelling units in Pre-IRA Scenario', '', '', ''],
    bin_number='auto',
    lower_percentile=lower_percentile,
    upper_percentile=upper_percentile,
    subplot_titles=['Fixed Discount Rate\n Low (2%)',
                    'Fixed Discount Rate\n Base (7%)',
                    'Fixed Discount Rate\n High (12%)',
                    'Variable Discount Rate\n Inverse to % AMI (7% to 45%)'],
    figure_size=(20, 10),  # Wide format for 4 panels
    sharex=False,  # Keep different scales to show full distributions
    sharey=True,   # Same y-scale for comparison
    color_code=f'base_{category}_fuel',
    show_legend=True
)

# Print comparison statistics
print("="*60)
print("Pre-IRA Scenario\nAdoption Feasibility under Different Discount Rate Assumptions")
print("="*60)

print_positive_percentages_complete(
    # df=df_outputs_mp8_ap2_FIXED_BASE,
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    column_names=[
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_low',
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_base',
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_high',
        f'preIRA_mp8_{category}_private_npv_moreWTP_variable'
    ],
    subplot_titles=['Fixed Discount Rate Low (2%)',
                    'Fixed Discount Rate Base (7%)',
                    'Fixed Discount Rate High (12%)',
                    'Variable Discount Rate Inverse to % AMI (7% to 45%)'],
    fuel_column=f'base_{category}_fuel'
)

fig_HEATING_preIRA_private_more_WTP_discount

In [ ]:
# Discount Rate Sensitivity Analysis
category = 'heating'

fig_HEATING_iraRef_private_more_WTP_discount = create_subplot_grid_histogram(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    # dataframe_indices=[0, 0],
    subplot_positions=[(0, 0), (0, 1), (0, 2), (0, 3)],
    x_cols=[
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_low',
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_base',
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_high',
        f'iraRef_mp8_{category}_private_npv_moreWTP_variable'
    ],
    x_labels=['Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]'
    ],
    y_labels=['Dwelling units in IRA-Reference Scenario', '', '', ''],
    bin_number='auto',
    lower_percentile=lower_percentile,
    upper_percentile=upper_percentile,
    subplot_titles=['Fixed Discount Rate\n Low (2%)',
                    'Fixed Discount Rate\n Base (7%)',
                    'Fixed Discount Rate\n High (12%)',
                    'Variable Discount Rate\n Inverse to % AMI (7% to 45%)'],
    figure_size=(20, 10),  # Wide format for 4 panels
    sharex=False,  # Keep different scales to show full distributions
    sharey=True,   # Same y-scale for comparison
    color_code=f'base_{category}_fuel',
    show_legend=True
)

# Print comparison statistics
print("="*60)
print("IRA-Reference Scenario\nAdoption Feasibility under Different Discount Rate Assumptions")
print("="*60)

print_positive_percentages_complete(
    # df=df_outputs_mp8_ap2_FIXED_BASE,
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    column_names=[
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_low',
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_base',
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_high',
        f'iraRef_mp8_{category}_private_npv_moreWTP_variable'
    ],
    subplot_titles=['Fixed Discount Rate Low (2%)',
                    'Fixed Discount Rate Base (7%)',
                    'Fixed Discount Rate High (12%)',
                    'Variable Discount Rate Inverse to % AMI (7% to 45%)'],
    fuel_column=f'base_{category}_fuel'
)

fig_HEATING_iraRef_private_more_WTP_discount

# Model Runtime

In [ ]:
# Get the current datetime again
end_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Calculate the elapsed time
elapsed_time = datetime.strptime(end_time, "%Y-%m-%d_%H-%M-%S") - datetime.strptime(start_time, "%Y-%m-%d_%H-%M-%S")

# Format the elapsed time
elapsed_seconds = elapsed_time.total_seconds()
elapsed_minutes = int(elapsed_seconds // 60)
elapsed_seconds = int(elapsed_seconds % 60)

# Print the elapsed time
print(f"The code took {elapsed_minutes} minutes and {elapsed_seconds} seconds to execute.")

In [ ]:
# print(f"The distribution of household income is: \n{df_outputs_mp8['household_income'].describe().round(0)}")

# print(f"\nThe distribution of percent AMI is: \n{df_outputs_mp8['percent_AMI'].describe().round(0)}")

# print(f"\nThe distribution of household variable discount rate is: \n{df_outputs_mp8['household_variable_discount_rate'].describe().round(0)}")

In [ ]:
# from cmu_tare_model.utils.create_sample_df import create_sample_df
# print_debug = True
# if PRINT_DEBUG:
#     # Create a sample dataframe for the heating category
#     df_sample_heating = create_sample_df(
#         df=df_outputs_mp8_inmap_FIXED_BASE,
#         include_groups=['base_equipment'],
#         categories=['heating'],
#         scenarios=['preIRA', 'iraRef'],
#         metrics=[],
#         mp_number=menu_mp,
#         regex_patterns=['household_income', 'income_level', 'percent_AMI', 'lmi_or_mui',
#                         'upgrade_heating', 'valid_fuel_heating', 'include_heating', 'discount',
#                         'heating_private_npv', 'heating_total_npv', 'heating_adoption_central_inmap_acs'
#                         ]
#     )
#     print(df_sample_heating)